<a href="https://colab.research.google.com/github/omsoni/llm-rag-work/blob/googlecolab-init/RAG_Pipeline_For_Medical_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --no-cache-dir --no-deps -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 105.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# For installing the libraries & downloading models from HF Hub
!pip install --upgrade huggingface_hub==0.35.3 pandas==2.2.2 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 chromadb==1.1.1  sentence-transformers==5.1.1  -q

In [ ]:
pip install diskcache llama-cpp-python==0.2.28 --no-deps --no-cache-dir -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 191.3 MB/s eta 0:00:00


In [ ]:
!pip install faiss-gpu-cu11

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 45.7 MB/s eta 0:00:00


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd
import numpy as np

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma
import faiss

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

In [ ]:
import textwrap
import warnings
warnings.filterwarnings('ignore')

#### function to pretty print a collection

In [ ]:
def pretty_print_doc_collection(relevant_document_chunks):
  for i, chunk in enumerate(relevant_document_chunks):
    print(f"── Chunk {i+1} ──────────────────────────────────────────")
    print(f"Page   : {chunk['metadata'].get('page', 'N/A')}")
    print(f"Score  : {chunk['score']:.4f}")
    print(f"Text   :")
    print(textwrap.fill(chunk['text'], width=80))
    print()

## Basic LLM Response evalution function

In [ ]:
def evaluate_response(response):
    return {
        "word_count":        len(response.split()),
        "has_disclaimer":    any(w in response.lower() for w in ["consult", "disclaimer"]),
        "is_structured":     any(c in response for c in ["1.", "•", "-", "\n"]),
        "mentions_treatment": any(w in response.lower() for w in ["intervention", "management", "treatment","therapy","medication","dose"]),
        "mentions_symptom":   any(w in response.lower() for w in ["recognition", "symptom","sign","present","diagnos"]),
    }

# Question Answering using LLM

Following Google Colab T4 friendly LLMs on huggingface were analyzed and compared:

| Model | Params | Domain | Strengths | Weaknesses | Colab T4 Friendly | Notes |
|------|------|------|------|------|------|------|
| meta-llama/Meta-Llama-3-8B-Instruct | 8B | General | Excellent reasoning, strong benchmarks, good instruction following | Not medical-specific | Runs with 4-bit quantization | Best overall |
| mistralai/Mistral-7B-Instruct-v0.2 | 7B | General | Fast inference, strong reasoning, widely used in RAG systems | Slightly weaker knowledge depth | Runs easily on T4 | Best for speed |
| epfl-llm/meditron-7b | 7B | Medical | Trained on PubMed and clinical texts | Weaker instruction following | Runs on T4 | Good domain baseline |
| BioMistral-7B | 7B | Medical | Biomedical pretraining improves performance over MediTron on some tasks | Some hallucination issues | Runs on T4 | Good medical candidate |
| OpenBioLLM-8B | 8B | Medical | Llama-3 based medical tuning | Less widely tested | Runs on T4 with quantization | Promising experimental |

In this step, the task requires the model to act as a medical assistant answering natural language questions. Based on Strengths and Weeknesses in the table, **meta-llama/Meta-Llama-3-8B-Instruct** is selected for response generation for its strong ability to generate:

* Structured explanations

* Follow prompts correctly

* Produce step-by-step reasoning


#### **Downloading the model from Hugging Face**

In [ ]:
model_name_or_path = "bartowski/Meta-Llama-3-8B-Instruct-GGUF"
model_basename = "Meta-Llama-3-8B-Instruct-Q4_K_M.gguf" # the model is in gguf format

In [ ]:
from huggingface_hub import login
login()

In [ ]:
bartowski_model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

Meta-Llama-3-8B-Instruct-Q4_K_M.gguf:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

#### **Model Configuration**
* Max context of 5000 tokens (Max allowed 8192 Tokens)
* Llama 3 8B has 32 transformer layers so 38 layers means every layer runs on GPU
* Process 512 tokens batch in parallel

In [ ]:
#uncomment the below snippet of code if the runtime is connected to GPU.
llm = Llama(
    model_path=bartowski_model_path,
    n_ctx=5000,
    n_gpu_layers=38,
    n_batch=512
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


#### **Model Parameters**

I have set model parameters for **deterministic** response.

* temperature=0 - always pick highest probability token
* top_p=0.95 - probability mass cutoff
* top_k=10 means: only consider top 10 tokens
* Max output tokens 256, Response gets truncated beyond that


#### Function to generate response

In [ ]:
def llm_response(query,max_tokens=256,temperature=0,top_p=0.95,top_k=10):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

#### **Declare Query Constants**

In [ ]:
from typing import Final
Query1: Final[str] = "What is the protocol for managing sepsis in a critical care unit?"
Query2: Final[str] = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
Query3: Final[str] = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
Query4: Final[str] = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
Query5: Final[str] = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"

#### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
response = llm_response(Query1)
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)

Llama.generate: prefix-match hit


 The protocol should include steps for identifying and treating patients with sepsis, as well as strategies for preventing sepsis.
The protocol for managing sepsis in a critical care unit typically includes the following steps:
1. Identification of Sepsis: Sepsis is identified by the presence of two or more of the following criteria:
* Temperature > 38°C (100.4°F) or < 36°C (96.8°F)
* Heart rate > 90 beats per minute
* Respiratory rate > 20 breaths per minute
* White blood cell count > 12,000 cells/mm³ or < 400 cells/mm³
* PaO2/FiO2 ratio < 300 mmHg
2. Initial Assessment: Upon identification of sepsis, the patient is assessed for severity using the Sequential Organ Failure Assessment (SOFA) score.
3. Fluid Resuscitation: Patients with sepsis are given a bolus of 30 mL/kg of lactated Ringer's solution or normal saline over 15-20 minutes to restore blood volume and perfusion.
4. Vasopressor Support: If the patient remains hypotensive after fluid resuscitation, vasopressors such as norepi

{'word_count': 160,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': False,
 'mentions_symptom': False}

#### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
response = llm_response(Query2)
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)

Llama.generate: prefix-match hit


 Appendicitis is a medical condition that occurs when the appendix becomes inflamed and fills with pus. The appendix is a small, finger-like pouch attached to the large intestine.
Common Symptoms of Appendicitis:
1. Severe abdominal pain: The most common symptom of appendicitis is severe abdominal pain that starts near the belly button and then moves to the lower right side of the abdomen.
2. Nausea and vomiting: Many people with appendicitis experience nausea and vomiting, which can be accompanied by fever, chills, and loss of appetite.
3. Abdominal tenderness: The abdomen may become tender to the touch, especially in the lower right quadrant.
4. Fever: A high fever is common in people with appendicitis, often above 100.4°F (38°C).
5. Loss of appetite: People with appendicitis may experience a loss of appetite and feel weak or fatigued.

Can Appendicitis be Cured via Medicine?
Appendicitis cannot be cured solely through medicine. Antibiotics may help alleviate symptoms and reduce the 

{'word_count': 187,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': False,
 'mentions_symptom': True}

#### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
response = llm_response(Query3)
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)

Llama.generate: prefix-match hit


 Hair loss can be a distressing experience, especially when it occurs suddenly and unexpectedly. In this article, we will explore some of the most common causes and effective treatments for sudden patchy hair loss.
Causes of Sudden Patchy Hair Loss:
1. Alopecia Areata: This is an autoimmune condition where the immune system attacks healthy hair follicles, leading to patchy hair loss.
2. Telogen Effluvium: This is a condition where there is an excessive shedding of hair due to hormonal changes, stress, or nutritional deficiencies.
3. Traction Alopecia: This occurs when hair is pulled too tightly, causing hair loss at the scalp.
4. Fungal Infections: Fungal infections like ringworm can cause patchy hair loss.
5. Nutritional Deficiencies: Lack of essential nutrients like iron, zinc, and biotin can contribute to hair loss.
6. Hormonal Imbalance: Hormonal changes during pregnancy, menopause, or thyroid disorders can lead to hair loss.
7. Stress: Physical or emotional stress can cause hair l

{'word_count': 180,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}

#### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
response = llm_response(Query4)
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)

Llama.generate: prefix-match hit


 The answer depends on the severity and location of the injury, as well as the individual's overall health. Here are some common treatment options:
1. Rest: In cases of mild traumatic brain injury (mTBI), rest is often recommended to allow the brain time to heal.
2. Medications: Pain relievers, anti-anxiety medications, and sleep aids may be prescribed to manage symptoms such as headaches, anxiety, or insomnia.
3. Physical therapy: Rehabilitation programs can help improve strength, balance, coordination, and cognitive function.
4. Occupational therapy: This type of therapy focuses on helping individuals with brain injuries regain daily living skills, such as dressing, grooming, and cooking.
5. Speech therapy: Speech therapists can help individuals with language processing difficulties or communication problems.
6. Cognitive rehabilitation: This type of therapy aims to improve memory, attention, problem-solving, and other cognitive functions.
7. Neurorehabilitation: This comprehensive a

{'word_count': 195,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

#### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
response = llm_response(Query5)
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)

 A fractured leg can be a serious injury that requires immediate attention and proper care to ensure optimal healing and minimize complications. Here's a comprehensive guide on how to handle such an emergency situation:

**Initial Response**

1. **Stop the bleeding**: Apply direct pressure to the wound using a clean cloth or gauze for at least 10-15 minutes to control any bleeding.
2. **Immobilize the leg**: Use a splint, sling, or crutches to immobilize the affected leg and reduce pain and swelling.
3. **Assess the injury**: Check for signs of shock, such as pale or cool skin, rapid pulse, and decreased blood pressure.

**Emergency Medical Treatment**

1. **Call 911 or local emergency services**: If you're in a remote area, call for medical help immediately.
2. **Transport to a hospital**: Get the injured person to a hospital or medical facility as quickly and safely as possible.
3. **Provide basic first aid**: Continue to apply direct pressure to any bleeding wounds and maintain immo

{'word_count': 190,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

### **Comments and Observations**

Following observations were made from the answers received from "Meta-Llama-3-8B-Instruct-GGUF"

* Without external knowledge source (RAG) the model relies on general training data. It does not have access Merck Manual and may use outdated medical material that it was trained on. E.g. The model seem to use outdated criterian for Sepsis identification.

* LLM does not provide any citation of its source of information.

* On Appendicitis query, model missed clinical signs, diagnostics, and treatment details. For query 3 ( patchy hair loss) model did reasonably good job in answering althought specifics for clinical diagnosis and treatment were missing.

* In query 5, LLM assumed there is bleeding but the query only mentioned fracture. Model changes the response structure, instead of earlier responses where bullet point response was given, query 5 returns response in sections.

This behavior is likley because model is anwering from training on various public documents, like Wikipedia, medical websites, health articles, research papers. Since model is providing answers that are not clinically accurate and complete, this approach can't be used as such.

Next step is to try to improve model response by using **prompt engineering**.

# Question Answering using LLM with Prompt Engineering

The goal here is to use **prompt engineering** to guide the model to reason, structure, and avoid hallucinations without adding external source of knowledge.

Idea is to create expert instruction prompts that forces the model to assume a professional role, provide structured answers by separating causes, symptoms, treatments and for safety add clinical caution and avoid unsupported claims.

We will apply **Instruction Prompting** and **Output Structuring** to structure the content for the purpose of being useful for human consumption. This will be different than structuring for machine consumption.

We will divide the prompt into **System Prompt** and **User prompt**. System prompt will be used for **Role prompting**, **Structured output**, **Safety constraints**

In addition to prompt engineering, we will try various hyperparameter configurations for our LLM, trying settings ranging from **Deterministic behavior** to **Exploratory response**. The parameters for all 5 settings are given in below table:

| Strategy | temperature | top_p | top_k | max_tokens | Use Case |
|---|---|---|---|---|---|
| Deterministic | 0 | 0.95 | 10 | 256 | Factual, consistent answers |
| Conservative | 0.1 | 0.9 | 20 | 256 | Slight variation, still safe |
| Balanced | 0.3 | 0.85 | 40 | 512 | General medical Q&A |
| Creative | 0.7 | 0.9 | 50 | 512 | Differential diagnosis |
| Exploratory | 1.0 | 0.95 | 100 | 1024 | Brainstorming, research |


* With Conservative Hyperparameter combination we will add **Constraint Prompting** by specifically adding some rules.
* With **Creative** and **Exploratory** Hyperparameter combination, we will add **Chain-of-Thought (CoT)** prompting to our template

In [ ]:
engineered_system_prompt = """
You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not replacement to a doctor's advise
- Recommend consulting a doctor

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation based on evidence
- Cause of the symptom
- Treatment protocol
- Bullet structured response
- Cite section titles if available
"""

In [ ]:
engineered_prompt_template = f"""
<SYSTEM>
{engineered_system_prompt}
</SYSTEM>

<USER>
###query
</USER>

<MEDICAL-ASSISTANT>
"""

In [ ]:
def get_engineered_prompt (query,prompt_template):
  engineered_prompt = prompt_template.replace('###query', query)
  return engineered_prompt

## **1. Deterministic Hyperparameter Optimization**
LLM transformer outputs can be made deterministic by setting temperature to 0, enabling greedy decoding (choosing the highest probability token). This is same confguration as our last run without prompt engineering.


* temperature=0 - always pick highest probability token
* top_p=0.95 - probability mass cutoff
* top_k=10 means: only consider top 10 tokens
* Max output tokens 256, Response gets truncated beyond that

In [ ]:
def llm_deterministic_response(prompt,max_tokens=256,temperature=0,top_p=0.95,top_k=10):
    model_output = llm(
      prompt=prompt,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

#### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
prompt = get_engineered_prompt(Query1, engineered_prompt_template)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_deterministic_response(prompt)
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)

******** PROMPT *****************

<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not replacement to a doctor's advise
- Recommend consulting a doctor

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation based on evidence
- Cause of the symptom
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>

<USER>
What is the protocol for managing sepsis in a critical care unit?
</USER>

<MEDICAL-ASSISTANT>

******** RESPONSE *****************
**Sepsis Management Protocol**

**Definition:** Sepsis is a l

{'word_count': 158,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

### Observations
if we compare this response with our first experiment without prompt engineering:

```
{'word_count': 160,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': False,
 'mentions_symptom': False}
```

The response with prompt engineering is given in **sections** as requested by our **system prompt** and mentions both treatment and symptomps. That is a **big improvement**.

#### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
prompt = get_engineered_prompt(Query2, engineered_prompt_template)
print("******** PROMPT *****************")
print(prompt)
response = llm_deterministic_response(prompt)
print("******** RESPONSE *****************")
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)

******** PROMPT *****************

<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not replacement to a doctor's advise
- Recommend consulting a doctor

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>

<USER>
What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?
</USER>

<MEDICAL-ASSISTANT>

******** RESPONSE *****************
**Appendicitis: Symptom

{'word_count': 162,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

### Observations
if we compare this response with our first experiment where no prompt engineering was applied:

```
{'word_count': 187,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': False,
 'mentions_symptom': True}
```

 The response with **prompt engineering** additionally mentions treatment. That is a ** improvement**.

#### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
prompt = get_engineered_prompt(Query3, engineered_prompt_template)
print("******** PROMPT *****************")
print(prompt)
response = llm_deterministic_response(prompt)
print("******** RESPONSE *****************")
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)

******** PROMPT *****************

<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>

<USER>
What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?
</USER>

<MEDICAL-ASSISTANT>



Llama.generate: prefix-match hit


******** RESPONSE *****************
**Sudden Patchy Hair Loss: Causes and Treatments**

**Causes:**
Patchy hair loss can occur due to various factors. Some common causes include:

• **Alopecia Areata**: An autoimmune disorder that causes the immune system to attack healthy hair follicles, leading to patchy hair loss.
• **Telogen Effluvium**: A condition where there is an excessive shedding of hair due to hormonal changes, stress, or nutritional deficiencies.
• **Fungal Infections**: Fungal infections like ringworm can cause patchy hair loss on the scalp.
• **Scalp Irritation**: Irritation from harsh chemicals, heat styling tools, or tight hairstyles can lead to patchy hair loss.

**Treatments:**
The most effective treatments for sudden patchy hair loss depend on the underlying cause. Some common treatment options include:

• **Topical Corticosteroids**: Creams or ointments containing corticosteroids can help reduce inflammation and promote hair growth.
• **Minoxidil**: A topical soluti

{'word_count': 168,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}

### Observations
if we compare this response with our first experiment where no prompt engineering was applied:

```
{'word_count': 180,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}
```

 The response with **prompt engineering** mentions both symptomps. That is an ** improvement**.

#### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
prompt = get_engineered_prompt(Query4, engineered_prompt_template)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_deterministic_response(prompt)
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)

******** PROMPT *****************

<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>

<USER>
What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?
</USER>

<MEDICAL-ASSISTANT>

******** RESPONSE *****************


Llama.generate: prefix-match hit


**Brain Injury Treatment**

A person who has sustained a physical injury to brain tissue may require immediate medical attention. The goal of treatment is to stabilize the patient and prevent further damage.

**Initial Assessment and Stabilization**

* Emergency responders will assess the patient's airway, breathing, and circulation (ABCs) to ensure they are stable.
* Patients with severe head injuries may be taken to an intensive care unit (ICU) for close monitoring.
* Imaging studies such as computed tomography (CT) or magnetic resonance imaging (MRI) scans may be ordered to evaluate the extent of brain damage.

**Medical Management**

* Pain management: Medications such as acetaminophen, ibuprofen, or opioids may be prescribed to manage pain and discomfort.
* Anti-seizure medications: Patients with a history of seizures or those who have suffered a severe head injury may be prescribed anti-seizure medications to prevent seizures.
* Blood pressure control: Patients with increased int

{'word_count': 188,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

### Observations

if we compare this response with our first experiment where no prompt engineering was applied:

```
{'word_count': 195,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}
```

 The response with **prompt engineering** provides structured response as asked in system prompt. Interestingly, LLM response without prompt engineering, also mentioned **Neurorehabilitation** and **Rehabilitation centers**. Since our prompt did not ask LLM about Rehabilitation as part of System Prompt, that category was not included by LLM.

#### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
prompt = get_engineered_prompt(Query5, engineered_prompt_template)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_deterministic_response(prompt)
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)

******** PROMPT *****************

<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>

<USER>
What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?
</USER>

<MEDICAL-ASSISTANT>

******** RESPONSE *****************


Llama.generate: prefix-match hit


**Fractured Leg: Precautions and Treatment Steps**

**Initial Assessment and Stabilization**

* Immediately stop any activity that may have caused the fracture.
* Apply a splint or immobilize the affected limb to prevent further injury.
* Monitor vital signs, including pulse, blood pressure, and oxygen saturation.

**Emergency Medical Care**

* If the person is experiencing severe bleeding, apply direct pressure to the wound using a clean cloth.
* If there are no signs of bleeding, but the person is in severe pain or has difficulty moving the affected limb, consider transporting them to an emergency medical facility for further evaluation and treatment.

**Treatment Protocol**

* Immobilize the fractured leg with a splint or cast to prevent further injury and promote healing.
* Administer pain management medication as prescribed by a healthcare provider.
* Monitor for signs of infection, such as redness, swelling, warmth, or pus around the wound.
* Consider administering antibiotics if

{'word_count': 205,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

if we compare this response with our first experiment where no prompt engineering was applied:


```
 {'word_count': 190,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

```

 The response with **prompt engineering** provided Care and Recovery section while LLM response without prompt engineering didn't cover this topic.


## **2. Conservative Hyperparamter Optimization**
A conservative hyperparameter configuration for LLM transformers focuses on stability, high precision, and avoiding over-fitting, particularly when fine-tuning. This approach often uses low learning rates, smaller batch sizes, and lower sampling temperatures.

* temperature=0.1 — Near Greedy
* top_k=20 — Slightly Wider Than Deterministic
* top_p=0.90 sits exactly between balanced and exploratory
* max_tokens=256 is kept small to force the LLM to be concise as directed in
**system prompt**

* With **Consevative Hyperparameter** combination we will add **Constraint Prompting** by specifically adding some rules.

In [ ]:
engineered_constraint_prompt_template = f"""
<SYSTEM>
{engineered_system_prompt}
</SYSTEM>
<CONSTRIANTS>
1. Answer in under 5 sentences
2. Cite the condition, cause, and treatment
3. End with: 'Consult a doctor for personal advice.'
</CONSTRIANTS>
<USER>
###query
</USER>
<MEDICAL-ASSISTANT>
"""

In [ ]:
def llm_conservative_response(query,max_tokens=256,temperature=0.1,top_p=0.90,top_k=20):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

#### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
prompt = get_engineered_prompt(Query1, engineered_constraint_prompt_template)
print(prompt)
print("******** RESPONSE *****************")
response = llm_conservative_response(prompt)
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)


<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>
<CONSTRIANTS>
1. Answer in under 5 sentences
2. Cite the condition, cause, and treatment
3. End with: 'Consult a doctor for personal advice.'
</CONSTRIANTS>
<USER>
What is the protocol for managing sepsis in a critical care unit?
</USER>
<MEDICAL-ASSISTANT>

******** RESPONSE *****************


Llama.generate: prefix-match hit


**Sepsis Management Protocol**

Sepsis is a life-threatening condition that occurs when an infection triggers an overwhelming inflammatory response throughout the body. The goal of sepsis management is to identify and treat the underlying cause, stabilize vital signs, and prevent organ dysfunction.

**Causes:**
• Infection (bacterial, viral, or fungal)
• Immune system dysregulation

**Treatment Protocol:**

1. **Initial Assessment:** Monitor vital signs, perform a physical examination, and obtain blood cultures.
2. **Fluid Resuscitation:** Administer IV fluids to maintain adequate perfusion.
3. **Antimicrobial Therapy:** Initiate broad-spectrum antibiotics or antifungals based on suspected pathogen.
4. **Supportive Care:** Provide oxygen therapy, mechanical ventilation if necessary, and manage pain and discomfort.

**Key Considerations:**

• Early recognition and treatment are crucial
• Monitor for signs of organ dysfunction (renal, respiratory, cardiovascular)
• Adjust treatment plan 

{'word_count': 170,
 'has_disclaimer': True,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

### Observations

if we compare this response with our first experiment where no prompt engineering was applied:

```
{'word_count': 183,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}
```
The response with **prompt engineering** provides symptoms and disclaimer.
The Conservative Hyperparamter Optimization gives model little more freedom (temperature=0.1,top_p=0.90) than **Determinitic** behavior. With **System Prompt** we added rule that answer be limited to 5 sentences. Clearlu LLM has responsed in a concise to the point response and ended with "Consult a doctor for personal advice."

#### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
prompt = get_engineered_prompt(Query2, engineered_constraint_prompt_template)
print(prompt)
print("******** RESPONSE *****************")
response = llm_conservative_response(prompt)
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)


<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not replacement to a doctor's advise
- Recommend consulting a doctor

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>
<CONSTRIANTS>
1. Answer in under 5 sentences
2. Cite the condition, cause, and treatment
3. End with: 'Consult a doctor for personal advice.'
</CONSTRIANTS>
<USER>
What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure

Llama.generate: prefix-match hit


**Appendicitis: Symptoms and Treatment**

Common symptoms of appendicitis include:

* Severe abdominal pain that starts near the belly button and moves to the lower right side
* Nausea and vomiting
* Loss of appetite
* Fever
* Abdominal tenderness

Appendicitis is typically caused by a blockage in the appendix, which can lead to inflammation and infection.

**Treatment:**

While antibiotics may be used to treat appendicitis, surgery is usually necessary to remove the inflamed appendix. The most common surgical procedure for appendicitis is an open appendectomy, where the surgeon makes a single incision in the abdomen to access the appendix. In some cases, laparoscopic surgery or robotic-assisted surgery may also be used.

**Consult a doctor for personal advice.**
</MEDICAL-ASSISTANT>assistant

Here's a revised version of the response that meets the constraints:

<SYSTEM>
...
</SYSTEM>

<CONSTRIANTS>
1. Answer in under 5 sentences
2. Cite the condition, cause, and treatment
3. End with:

{'word_count': 167,
 'has_disclaimer': True,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

### Observations

For Query2, **prompt engineering** and configuration modified behaviour of LLM exactly same as for Query1. ### Observations

if we compare this response with our first experiment where no prompt engineering was applied:

```
{'word_count': 187,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': False,
 'mentions_symptom': True}
```

The response with **prompt engineering** provides symptoms and disclaimer.  The response with **prompt engineering** provides symptoms and disclaimer.
The Conservative Hyperparamter Optimization gives model little more freedom (temperature=0.1,top_p=0.90) than **Determinitic** behavior. With **System Prompt** we added rule that answer be limited to 5 sentences. Clearlu LLM has responsed in a concise to the point response and ended with "Consult a doctor for personal advice."
It is important to notice that LLM closed the MEDICAL-ASSISTANT tag in the response that I was expecting it to close.

#### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
prompt = get_engineered_prompt(Query3, engineered_constraint_prompt_template)
print(prompt)
print("******** RESPONSE *****************")
response = llm_conservative_response(prompt)
print(response);
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)


<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not replacement to a doctor's advise
- Recommend consulting a doctor

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>
<CONSTRIANTS>
1. Answer in under 5 sentences
2. Cite the condition, cause, and treatment
3. End with: 'Consult a doctor for personal advice.'
</CONSTRIANTS>
<USER>
What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localize

Llama.generate: prefix-match hit


**Sudden Patchy Hair Loss: Causes and Treatments**

Patchy hair loss, also known as alopecia areata, is an autoimmune condition where the immune system mistakenly attacks healthy hair follicles. This can lead to sudden, patchy bald spots on the scalp.

**Causes:**
• **Autoimmune response**: The immune system mistakenly identifies healthy hair follicles as foreign and attacks them.
• **Stress**: Physical or emotional stress can trigger an autoimmune response.
• **Genetics**: Family history of alopecia areata increases the risk.

**Treatments:**

• **Topical corticosteroids**: Apply a mild steroid cream to reduce inflammation and promote hair growth.
• **Minoxidil**: Use over-the-counter (OTC) minoxidil solutions or foams to stimulate hair growth.
• **Phototherapy**: Exposure to ultraviolet (UV) light, such as narrowband UVB or PUVA, can help restore hair growth.

**Consult a doctor for personal advice.**
</MEDICAL-ASSISTANT>

```
```python
import re
from collections import defaultdict
f

{'word_count': 152,
 'has_disclaimer': True,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}

### Observations
Comparing above response with our first experiment where no prompt engineering was applied:

```
{'word_count': 180,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}
```

 The response with **prompt engineering** adds disclaimer and returns a structured response. That is improvement over the base scenario without prompt engineering. The instruction in prompt to give answer in 5 sentences seems to shorten the response cutting the information.

#### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
prompt = get_engineered_prompt(Query4, engineered_constraint_prompt_template)
print(prompt)
print("******** RESPONSE *****************")
response = llm_conservative_response(prompt)
print(response);
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)


<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not replacement to a doctor's advise
- Recommend consulting a doctor

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>
<CONSTRIANTS>
1. Answer in under 5 sentences
2. Cite the condition, cause, and treatment
3. End with: 'Consult a doctor for personal advice.'
</CONSTRIANTS>
<USER>
What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in t

Llama.generate: prefix-match hit


**Brain Injury Treatment**

A traumatic brain injury (TBI) occurs when the brain is damaged due to external forces, such as a blow to the head or penetrating head trauma. The severity and location of the injury determine the extent of damage.

**Treatment:**

* **Acute Care:** Initial treatment focuses on stabilizing the patient's condition, managing symptoms, and preventing further injury.
	+ Medications: Pain management, anti-seizure medications, and sedatives as needed
	+ Monitoring: Close observation for signs of increased intracranial pressure (ICP)
	+ Surgery: If necessary, to relieve pressure or repair damaged tissue
* **Rehabilitation:** After initial stabilization, rehabilitation aims to improve cognitive, physical, and emotional function.
	+ Physical Therapy: To regain motor skills and strength
	+ Occupational Therapy: To relearn daily activities and adapt to new abilities
	+ Speech Therapy: To address communication difficulties

**Consult a doctor for personal advice.**
</ME

{'word_count': 165,
 'has_disclaimer': True,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

### Observations

Comparing above response with our first experiment where no prompt engineering was applied:

```
{'word_count': 195,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}
```

 The response with **prompt engineering** does little worse. The instruction in prompt to give answer in 5 sentences seems to cut off the response limiting the information provided by LLM. This however will be tested in our next scenario as we change the prompt to answer in 10 sentences or less.

#### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
prompt = get_engineered_prompt(Query5, engineered_constraint_prompt_template)
print(prompt)
print("******** RESPONSE *****************")
response = llm_conservative_response(prompt)
print(response);
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)


<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not replacement to a doctor's advise
- Recommend consulting a doctor

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>
<CONSTRIANTS>
1. Answer in under 5 sentences
2. Cite the condition, cause, and treatment
3. End with: 'Consult a doctor for personal advice.'
</CONSTRIANTS>
<USER>
What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking 

Llama.generate: prefix-match hit


**Fractured Leg: Precautions and Treatment**

A fractured leg, also known as a lower extremity fracture, is a common injury that can occur due to trauma or falls. As a medical assistant, it's essential to provide accurate information on necessary precautions and treatment steps for proper care and recovery.

**Causes:** Fractures of the leg can be caused by various factors such as:
	* Trauma (e.g., car accidents, falls)
	* Sports injuries
	* Osteoporosis

**Symptoms:**
	* Severe pain in the affected area
	* Swelling and bruising
	* Deformity or abnormal alignment of the leg
	* Limited mobility

**Treatment:** The primary goal is to stabilize the fracture, relieve pain, and promote healing. Treatment may include:
	* Immobilization with a cast, splint, or brace
	* Pain management with medication (e.g., acetaminophen, ibuprofen)
	* Physical therapy to maintain range of motion and strength

**Precautions:**
	* Avoid putting weight on the affected leg
	* Elevate the injured area above heart

{'word_count': 165,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

Comparing above response with our first experiment where no prompt engineering was applied:

```
{'word_count': 183,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}
```

The response with **prompt engineering** provides very structured response. The response covers all required sections except the disclaimer. Overall the LLM does better with this prompt. So far we have noticed the almost always **prompt engineering** does better than just sending the plain question to LLM.


## **3. Balanced Hyperparamter Optimization**

Balanced hyperparameter combinations for Large Language Model (LLM) transformer training and fine-tuning focus on achieving optimal performance, stability, and computational efficiency without overfitting or excessive, costly experimentation:

At 0.3 the model strongly favors the top token but occasionally picks the second or third best — giving slight variation while staying accurate.

top_k=40 — Medium Candidate Pool
top_k opens door to 40 tokens
top_p=0.85 closes it back to ~2-3 high quality tokens
More restrictive than exploratory (0.95) but less than deterministic


In [ ]:
def llm_balanced_response(query,max_tokens=512,temperature=0.3,top_p=0.85,top_k=40):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

In [ ]:
engineered_balanced_prompt_template = f"""
<SYSTEM>
{engineered_system_prompt}
</SYSTEM>
<CONSTRIANTS>
1. Answer in under 10 sentences
2. Cite the condition, cause, and treatment
3. End with: 'Consult a doctor for personal advice.'
</CONSTRIANTS>
<USER>
###query
</USER>
<MEDICAL-ASSISTANT>
"""

#### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
prompt = get_engineered_prompt(Query1, engineered_balanced_prompt_template)
print(prompt)
print("******** RESPONSE *****************")
response = llm_balanced_response(prompt)
print(response);
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)


<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not replacement to a doctor's advise
- Recommend consulting a doctor

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>
<CONSTRIANTS>
1. Answer in under 10 sentences
2. Cite the condition, cause, and treatment
3. End with: 'Consult a doctor for personal advice.'
</CONSTRIANTS>
<USER>
What is the protocol for managing sepsis in a critical care unit?
</USER>
<MEDICAL-ASSISTANT>

******** RESPONS

Llama.generate: prefix-match hit


**Sepsis Management Protocol**

Sepsis is a life-threatening condition that occurs when an infection triggers a systemic inflammatory response. In a critical care unit, early recognition and prompt management are crucial to improve patient outcomes.

**Causes:**
Sepsis can be caused by various infections, including pneumonia, urinary tract infections, and intra-abdominal infections.

**Symptoms:**
Common symptoms of sepsis include fever, tachycardia, tachypnea, altered mental status, and decreased blood pressure.

**Treatment Protocol:**

1. **Initial Assessment:** Perform a thorough physical examination, obtain a complete medical history, and order laboratory tests to identify the source of infection.
2. **Fluid Resuscitation:** Administer fluids to maintain adequate perfusion and prevent organ dysfunction.
3. **Antimicrobial Therapy:** Initiate broad-spectrum antibiotics based on suspected pathogens and adjust as necessary based on culture results.
4. **Supportive Care:** Provide oxy

{'word_count': 347,
 'has_disclaimer': True,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

### Observations

Comparing this response with our first experiment where no prompt engineering was applied:


```
{'word_count': 183,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}
```

 The response with **prompt engineering** provides the desired output. There is minor hyperparameter changes significant one is max_tokens=512 which allows for longer response. In addition in the system prompt we have asked LLM to now respond in 10 sentences instead of 5 in **Convervative behavior** settings. But we can tell that LLM does not know when to stop and it keeps generating. This seems like my **prompt** has some problem in it. It likely does not like the xml tags which was my attempt to comform with Llama 3 prompt format. It seems like the LLM is evaluating and justying its own response to rules within the <Constraints> tags. Since now the max tokens is 512, model is able to output these additional tokens to respond against rules in <Constraints>.


#### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
prompt = get_engineered_prompt(Query2, engineered_balanced_prompt_template)
print(prompt)
print("******** RESPONSE *****************")
response = llm_balanced_response(prompt)
print(response);
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)


<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not replacement to a doctor's advise
- Recommend consulting a doctor

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>
<CONSTRIANTS>
1. Answer in under 10 sentences
2. Cite the condition, cause, and treatment
3. End with: 'Consult a doctor for personal advice.'
</CONSTRIANTS>
<USER>
What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedur

Llama.generate: prefix-match hit


**Appendicitis: Symptoms and Treatment**

**Symptoms:**
Common symptoms of appendicitis include:
• Sudden onset of severe abdominal pain, usually starting near the belly button (McBurney's point) and then migrating to the lower right abdomen
• Nausea and vomiting
• Loss of appetite
• Fever
• Abdominal tenderness

**Cause:**
Appendicitis occurs when the appendix becomes inflamed and fills with pus, leading to a blockage in the intestine.

**Treatment:**

* **Medication:** Antibiotics may be prescribed to treat appendicitis, but they are not a cure. They can help alleviate symptoms and prevent complications.
* **Surgical Procedure:** If antibiotics do not work or if there is a high risk of rupture, surgery is necessary to remove the appendix (appendectomy).
	+ Open appendectomy: A traditional surgical procedure where the abdomen is opened to access the appendix.
	+ Laparoscopic appendectomy: A minimally invasive surgical procedure using small incisions and a camera.

**Important:** Appen

{'word_count': 326,
 'has_disclaimer': True,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

### Observations

Comparing above response with our first experiment where no prompt engineering was applied:

```
{'word_count': 187,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': False,
 'mentions_symptom': True}
```

 The response with **prompt engineering** is significantly better. Clearly LLM is generating response and then evaluating its own response against constraints in the **system prompt**. which is cool. We will get rid of the tags in next scenario and see if that helps the LLM calm down.

#### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
prompt = get_engineered_prompt(Query3, engineered_balanced_prompt_template)
print(prompt)
print("******** RESPONSE *****************")
response = llm_balanced_response(prompt)
print(response);
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)


<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not replacement to a doctor's advise
- Recommend consulting a doctor

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>
<CONSTRIANTS>
1. Answer in under 10 sentences
2. Cite the condition, cause, and treatment
3. End with: 'Consult a doctor for personal advice.'
</CONSTRIANTS>
<USER>
What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localiz

Llama.generate: prefix-match hit


**Sudden Patchy Hair Loss: Causes and Treatments**

Patchy hair loss, also known as alopecia areata, is an autoimmune condition characterized by sudden onset of localized bald spots on the scalp. The exact cause is unknown, but it's believed to be triggered by a combination of genetic, environmental, and hormonal factors.

**Possible Causes:**

• **Genetic predisposition:** Family history of alopecia areata or other autoimmune disorders
• **Stress:** Physical or emotional stress can trigger an immune response leading to hair loss
• **Hormonal imbalance:** Hormonal changes during pregnancy, menopause, or thyroid disorders
• **Infections:** Bacterial or fungal infections affecting the scalp

**Treatments:**

• **Topical corticosteroids:** Creams or ointments applied directly to the affected area to reduce inflammation and promote hair growth
• **Minoxidil:** Over-the-counter (OTC) solution applied topically to stimulate hair growth and slow down hair loss
• **Phototherapy:** Exposure to 

{'word_count': 337,
 'has_disclaimer': True,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}

### Observations

Comparing above response with our first experiment where no prompt engineering was applied:

```
{'word_count': 180,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}
```

The response with **prompt engineering** is significantly better. Symptoms are missing. It seems like hair loss symptoms are self explainatory. LLM still continue to generate secondary response to constraints.

#### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
prompt = get_engineered_prompt(Query4, engineered_balanced_prompt_template)
print(prompt)
print("******** RESPONSE *****************")
response = llm_balanced_response(prompt)
print(response);
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)


<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not replacement to a doctor's advise
- Recommend consulting a doctor

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>
<CONSTRIANTS>
1. Answer in under 10 sentences
2. Cite the condition, cause, and treatment
3. End with: 'Consult a doctor for personal advice.'
</CONSTRIANTS>
<USER>
What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in 

Llama.generate: prefix-match hit


**Brain Injury Treatment**

A traumatic brain injury (TBI) occurs when the brain is damaged due to external forces, such as a blow to the head, penetration by an object, or sudden acceleration/deceleration. The severity and location of the injury determine the extent of damage.

**Causes:**
• Blunt trauma
• Penetrating injuries
• Falls
• Motor vehicle accidents

**Symptoms:**
• Headache
• Confusion
• Dizziness
• Loss of consciousness
• Memory loss
• Mood changes

**Treatment:**

* Emergency care:
	+ Stabilize the patient's airway, breathing, and circulation (ABCs)
	+ Monitor vital signs and neurological status
* Medical management:
	+ Pain relief medication
	+ Anti-seizure medications if necessary
	+ Anticonvulsants for seizure control
	+ Rehabilitation therapy to improve cognitive and motor function

**Rehabilitation:**
• Physical therapy to regain strength, balance, and coordination
• Occupational therapy to relearn daily activities and skills
• Speech therapy to address communicatio

{'word_count': 333,
 'has_disclaimer': True,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

### Observations

Comparing above response with our first experiment where no prompt engineering was applied:

```
{'word_count': 195,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}
```

 The response with **prompt engineering** is significantly better. Symptoms are missing. It seems like hair loss symptoms are self explainatory. LLM continue to generate secondary response to constraints.

#### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
prompt = get_engineered_prompt(Query5, engineered_balanced_prompt_template)
print(prompt)
print("******** RESPONSE *****************")
response = llm_balanced_response(prompt)
print(response);
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)


<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not replacement to a doctor's advise
- Recommend consulting a doctor

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>
<CONSTRIANTS>
1. Answer in under 10 sentences
2. Cite the condition, cause, and treatment
3. End with: 'Consult a doctor for personal advice.'
</CONSTRIANTS>
<USER>
What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking

Llama.generate: prefix-match hit


**Fractured Leg: Precautions and Treatment**

A fractured leg, also known as a lower extremity fracture, is a common injury that can occur due to trauma or falls. As a medical assistant, it's essential to provide proper care and treatment to ensure optimal recovery.

**Causes:**
The most common causes of fractured legs are:
• Trauma or falls
• Sports-related injuries
• Motor vehicle accidents

**Symptoms:**
Common symptoms of a fractured leg include:
• Severe pain
• Swelling and bruising
• Deformity or abnormal alignment
• Difficulty moving the affected limb

**Treatment:**

1. **Initial Care:** Apply ice packs to reduce swelling, elevate the injured leg above heart level, and immobilize it with a splint or cast.
2. **Medical Evaluation:** Seek immediate medical attention for proper evaluation and treatment by a healthcare professional.
3. **Imaging Studies:** X-rays, CT scans, or MRI may be ordered to confirm the fracture and determine its severity.
4. **Surgery:** In some cases, surg

{'word_count': 347,
 'has_disclaimer': True,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

Comparing above response with our first experiment where no prompt engineering was applied:

```
{'word_count': 183,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}
```

 The response with **prompt engineering** is significantly better. Symptoms are missing. It seems like hair loss symptoms are self explainatory. LLM continue to generate secondary response to constraints insisting "The Medical Assistant's response adheres to the constraints set forth"


## **4. Creative Hyperparameter Optimization**

Below configuration produces more creative responses because the sampling parameters increase diversity in token selection rather than always choosing the most probable next token.

* With temperature = 0.7, probability differences between tokens are reducedand lower-ranked tokens get more chance to be selected.
* Top_p 0f .9 doesn't forces model to pick the highest probability token allowing more choices but avoiding meaningless tokens

In [ ]:
def llm_creative_response(query,max_tokens=512,temperature=0.7,top_p=0.9,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

In [ ]:
#Notice no constraints.
engineered_chain_of_thought_prompt_template = f"""
<SYSTEM>
{engineered_system_prompt}
Let's think through this step by step.
</SYSTEM>
<USER>
###query
</USER>

<MEDICAL-ASSISTANT>
"""

#### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
prompt = get_engineered_prompt(Query1, engineered_chain_of_thought_prompt_template)
print(prompt)
print("******** RESPONSE *****************")
response = llm_creative_response(prompt)
print(response);
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)


<SYSTEM>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not replacement to a doctor's advise
- Recommend consulting a doctor

Structure answers using sections when appropriate.
Focus on clarity and clinical relevance.
Provide:
- Clinical explanation
- Treatment protocol
- Bullet structured response
- Cite section titles if available

</SYSTEM>
<THINK>
Let's think through this step by step.
</THINK>
<USER>
What is the protocol for managing sepsis in a critical care unit?
</USER>

<MEDICAL-ASSISTANT>

******** RESPONSE *****************


Llama.generate: prefix-match hit


**Sepsis Management Protocol**

As a Medical Assistant, I'd like to provide you with a structured response on managing sepsis in a critical care unit.

**Clinical Explanation:**
Sepsis is a life-threatening condition characterized by an uncontrolled inflammatory response to infection. Early recognition and treatment are crucial to improve patient outcomes.

**Treatment Protocol:**

1. **Initial Assessment:**
	* Obtain a thorough medical history, including recent infections, surgeries, or trauma.
	* Perform a physical examination, focusing on vital signs, cardiac function, and lung sounds.
2. **Diagnostic Evaluation:**
	* Complete blood count (CBC) to assess for leukocytosis or leukopenia.
	* Blood cultures to identify the causative pathogen.
	* Chest X-ray or computed tomography (CT) scan to evaluate lung involvement.
3. **Fluid Management:**
	* Administer intravenous fluids (IVFs) to maintain adequate hydration and blood pressure.
	* Monitor central venous oxygen saturation (ScvO2) an

{'word_count': 324,
 'has_disclaimer': True,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

Comparing above response with our first experiment where no prompt engineering was applied:

```
{'word_count': 160,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': False,
 'mentions_symptom': False}
```

 The response with **prompt engineering** provides a lot of information. Seems like with higher temperature LLM is going off script. Before next experiment, we will make changes to prompts to see if we can make LLM not comment on its own response.

 * Tell LLM to provide answer once not evaluate or comment on response.
 * Shift to Llama3 prompt format


In [ ]:
engineered_system_prompt_v2 = """
You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not a replacement for a doctor
- Recommend consulting a doctor

Provide:
- Clinical explanation based on evidence
- Cause of the symptom
- Treatment protocol
- Bullet structured response

IMPORTANT: Provide your answer ONCE. Do not evaluate or comment on your response.
"""

In [ ]:
## Llama 3 format
def get_cot_prompt(query):
    return (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n"
        f"{engineered_system_prompt_v2}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n"
        f"{query}\n"
        f"Let's think through this step by step.<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>"
    )

#### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
prompt = get_cot_prompt(Query2)
print(prompt)
print("******** RESPONSE *****************")
response = llm_creative_response(prompt)
print(response);
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not a replacement for a doctor
- Recommend consulting a doctor

Provide:
- Clinical explanation based on evidence
- Cause of the symptom
- Treatment protocol
- Bullet structured response

IMPORTANT: Provide your answer ONCE. Do not evaluate or comment on your response.
<|eot_id|><|start_header_id|>user<|end_header_id|>
What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?
Let's think through this step by step.<|eot_id|><|

Llama.generate: prefix-match hit




**Disclaimer:** I am a Medical Assistant and not a replacement for a doctor. It is essential to consult with a healthcare professional for personalized medical advice.

**Symptoms of Appendicitis:**

Appendicitis typically presents with a combination of the following symptoms:

* Severe abdominal pain, usually starting near the belly button and moving to the lower right abdomen
* Nausea and vomiting
* Fever (usually above 101.5°F or 38.6°C)
* Loss of appetite
* Abdominal swelling
* Guarding (tensing) of the muscles in the abdomen when pressure is applied

**Cause of Appendicitis:**

Appendicitis occurs when the appendix, a small pouch attached to the large intestine, becomes inflamed and fills with pus. The exact cause is often unclear, but it may be related to:

* Blockage by feces, food, or other debris
* Infection by bacteria, such as E. coli
* Trauma to the abdomen

**Treatment Protocol:**

Appendicitis cannot be cured solely through medicine. Surgery is usually necessary to remo

{'word_count': 353,
 'has_disclaimer': True,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

Comparing above response with our earlier experiment where no prompt engineering was applied:

```
 {'word_count': 187,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': False,
 'mentions_symptom': True}
```

 The response with **prompt engineering** provides perfect response.
 And there is no confusion with LLM because of tags in our prompt and it did listen to our IMPORTANT instruction to not to evaluate or rate its own response. This is the best response we have received so far.


#### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
prompt = get_cot_prompt(Query3)
print(prompt)
print("******** RESPONSE *****************")
response = llm_creative_response(prompt)
print(response);
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not a replacement for a doctor
- Recommend consulting a doctor

Provide:
- Clinical explanation based on evidence
- Cause of the symptom
- Treatment protocol
- Bullet structured response

IMPORTANT: Provide your answer ONCE. Do not evaluate or comment on your response.
<|eot_id|><|start_header_id|>user<|end_header_id|>
What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?
Let's t

Llama.generate: prefix-match hit




**Disclaimer:** As a Medical Assistant, I am not a replacement for a doctor. Please consult a healthcare professional for personalized advice.

**Patchy Hair Loss: A Step-by-Step Guide**

**Causes of Patchy Hair Loss:**

1. **Alopecia Areata:** An autoimmune condition where the immune system attacks hair follicles, leading to patchy hair loss.
2. **Telogen Effluvium:** When a sudden shock or stress causes excessive shedding and hair thinning, often seen in cases like childbirth, illness, or medication changes.
3. **Thyroid Disorders:** Hypothyroidism (underactive thyroid) or hyperthyroidism (overactive thyroid) can cause patchy hair loss due to hormonal imbalances.
4. **Fungal Infections:** Fungal infections like ringworm or scalp psoriasis can lead to patchy hair loss and scaling.
5. **Genetic Predisposition:** Family history of patchy hair loss or baldness.

**Effective Treatments:**

1. **Topical Corticosteroids:** Creams or ointments applied directly to the affected area to reduc

{'word_count': 333,
 'has_disclaimer': True,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

Comparing above response with our first experiment where no prompt engineering was applied:

```
 {'word_count': 180,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}
```

 The response with **prompt engineering** covers all required sections and is behaving way better providing expected information and not going off script.


#### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
prompt = get_cot_prompt(Query4)
print(prompt)
print("******** RESPONSE *****************")
response = llm_creative_response(prompt)
print(response);
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not a replacement for a doctor
- Recommend consulting a doctor

Provide:
- Clinical explanation based on evidence
- Cause of the symptom
- Treatment protocol
- Bullet structured response

IMPORTANT: Provide your answer ONCE. Do not evaluate or comment on your response.
<|eot_id|><|start_header_id|>user<|end_header_id|>
What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?
Let's think through this step b

Llama.generate: prefix-match hit




**Disclaimer:** As a medical assistant, I am not a replacement for a doctor. It is essential to consult a healthcare professional for personalized guidance and treatment.

**Step-by-Step Treatment Protocol:**

1. **Initial Assessment:**
	* The individual should seek immediate medical attention at an emergency department or urgent care center.
	* A thorough neurological examination will be conducted to assess the extent of the injury, including:
		+ Glasgow Coma Scale (GCS) assessment
		+ Imaging studies (CT or MRI scans)
		+ Neurological function tests (e.g., motor, sensory, cognitive)
2. **Acute Management:**
	* Stabilization and management of vital signs (BP, HR, O2 saturation, etc.)
	* Prevention of secondary brain injury:
		+ Control of blood pressure
		+ Maintenance of normal body temperature
		+ Avoidance of hypoxia and hypercapnia
3. **Pharmacological Interventions:**
	* Pain management (e.g., acetaminophen, NSAIDs)
	* Sedation (e.g., benzodiazepines) for agitation or anxiety


{'word_count': 318,
 'has_disclaimer': True,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

Comparing above response with our first experiment where no prompt engineering was applied:

```
 {'word_count': 195,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}
```

 The response with **prompt engineering** covers all required sections and is responding way better. We will skip Query 5 and go to our next experiment with even higher temperature to get more exploratory behavior which can be really useful for acedemic use cases.



## **5. Exploratory Hyperparameter Optimization**

* temperature=1.0 - use distribution as-is, sampling reflects raw model probabilities
* top_p=0.95 - cut off probability
* top_k=100 - consider top 100 choices
* max_tokens=1024 - so that detailed answer for exploration does not get cutout

These parameters together essentially remove most restrictions on the model's output generation. Here's what each does at these values:
Dosage or drug interaction queries (too risky with hallucinations)
Patient-facing responses (inconsistency is dangerous)

Since accuracy drops, for medical AI, we should use exploratory only as a brainstorming layer, then validate outputs with a deterministic pass at temperature=0.

In [ ]:
def llm_exploratory_response(query,max_tokens=1024,temperature=1,top_p=0.95,top_k=100):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

#### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
prompt = get_cot_prompt(Query1)
print(prompt)
print("******** RESPONSE *****************")
response = llm_exploratory_response(prompt)
print(response);
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not a replacement for a doctor
- Recommend consulting a doctor

Provide:
- Clinical explanation based on evidence
- Cause of the symptom
- Treatment protocol
- Bullet structured response

IMPORTANT: Provide your answer ONCE. Do not evaluate or comment on your response.
<|eot_id|><|start_header_id|>user<|end_header_id|>
What is the protocol for managing sepsis in a critical care unit?
Let's think through this step by step.<|eot_id|><|start_header_id|>assistant<|end_header_id|>
******** RESPONSE *****************

Llama.generate: prefix-match hit




**Disclaimer:** I am a Medical Assistant and am not a replacement for a doctor. Please consult a healthcare professional for personalized medical advice.

**Step 1: Early Recognition and Diagnosis**

* Sepsis is a life-threatening condition that requires prompt recognition and diagnosis.
* Look for early warning signs:
	+ Fever above 38°C (100.4°F) or hypothermia below 36°C (96.8°F)
	+ Tachycardia (heart rate >120 beats per minute)
	+ Tachypnea (respiratory rate >20 breaths per minute)
	+ Altered mental status
	+ Decreased blood pressure
* Use the Sepsis-3 criteria:
	+ QuickSOFA score ≥ 2 or qSOFA score ≥ 1

**Step 2: Initial Assessment and Resuscitation**

* Activated clotting time (ACT) and platelet count to rule out disseminated intravascular coagulation
* Blood culture from peripheral vein, central venous catheter, or other relevant sites
* Urine output monitoring
* Insertion of urinary catheter if necessary
* Administer broad-spectrum antibiotics within 1-3 hours after suspicion

{'word_count': 667,
 'has_disclaimer': True,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

Comparing above response with our first experiment where no prompt engineering was applied:

```
{'word_count': 160,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': False,
 'mentions_symptom': False}
```

 The response with **prompt engineering** covers all expectations provided in system and user promt and is responding with detailed information. This is great from a LLM that is not tied to medical domain and is trained for general purpose. It is doing all that was expected from it. It is showing

* strong reasoning ability

* Follows instructions given in the prompt

* gives complex explanations in logical structure

Also, we noticed it is important to configure max tokens to so the LLM does not cut off the response. Format our **prompts** properly and set other hyperparameter properly. This learning will be used in our next effort to build our actual Medical Assistant.

I am going to run one more query on this and skip the other three.

#### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
prompt = get_cot_prompt(Query2)
print(prompt)
print("******** RESPONSE *****************")
response = llm_exploratory_response(prompt)
print(response);
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a highly knowledgeable medical assistant trained to provide
accurate, clear, and structured medical explanations.

Your responses should:
- Use medically correct terminology
- Be structured and easy to read
- Clearly explain symptoms, causes, and treatments when applicable
- Avoid unsupported medical claims
- Cite medical sources when appropriate with dates if possible
- Add disclaimer that Medical Assistant is not a replacement for a doctor
- Recommend consulting a doctor

Provide:
- Clinical explanation based on evidence
- Cause of the symptom
- Treatment protocol
- Bullet structured response

IMPORTANT: Provide your answer ONCE. Do not evaluate or comment on your response.
<|eot_id|><|start_header_id|>user<|end_header_id|>
What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?
Let's think through this step by step.<|eot_id|><|

Llama.generate: prefix-match hit




**Disclaimer:** A medical assistant is not a replacement for a doctor. This response provides general information and should not replace a professional medical consultation.

**Common Symptoms of Appendicitis:**

* Sudden onset of severe abdominal pain, usually on the lower right side (McBurney's point)
* Nausea and vomiting
* Loss of appetite
* Fever
* Abdominal tenderness or swelling
* Pain that worsens with movement or coughing

**Cause of Appendicitis:**

Appendicitis is typically caused by a blockage of the appendix, usually due to:

* Inflammatory bowel disease (IBD)
* Foreign object lodgment
* Intussusception (a condition where one portion of the intestine telescopes into another)
* Trauma
* Cancer

**Treatment Protocol:**

If appendicitis is suspected, immediate medical attention is necessary. If left untreated, the appendix can rupture, leading to serious complications and potentially life-threatening infections.

**Medication:**

Appendicitis cannot be cured solely through 

{'word_count': 712,
 'has_disclaimer': True,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

# **RAG Architecture**

In **Retrieval-Augmented Generation (RAG)**, the **retriever** component is responsible for fetching relevant documents or context from a knowledge base to enhance the model's generation. Here are the key steps involved in the **retriever part**:

1. **Chunking**:  
   - Split large documents into smaller, manageable chunks while maintaining context (e.g., using `RecursiveCharacterTextSplitter`).

2. **Embedding Generation**:  
   - Convert each text chunk into a **vector embedding** using a pre-trained embedding model.

3. **Vector Storage**:  
   - Store the embeddings in a **vector database** (e.g., Chroma, Pinecone, or FAISS) for efficient similarity search.

4. **Similarity Search**:  
   - Perform a **similarity search** in the vector store to find the most relevant document chunks based on the query embedding.
5. **Prompt Engineering with Content** Pass the context retrieved from vector database for the query in the prompt

6. Get response back from LLM

<img src="https://drive.google.com/uc?id=1-0_pjR7gg07Zxt7uiZFHP-xpWqBfeqT_">

### Data Preparation for RAG

In [ ]:
# uncomment and run the below code snippets if the dataset is present in the Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Using hybrid approach to read PDF to ensure both MetaData and Text are extracted. This approach ensure RAG ready extraction (semantic chunking, traceable metadata such as page, sections).

#### Normalize whitespace

PDF extraction usually causes random line breaks, broken paragraphs, double spaces

In [ ]:
import re

def clean_text(text):
    text = re.sub(r'-\n', '', text)          # fix hyphenated line breaks
    text = re.sub(r'\n+', ' ', text)         # collapse newlines
    text = re.sub(r'\s+', ' ', text)         # collapse whitespace
    return text.strip()

In [ ]:
def load_pdf_with_metadata(path):
    doc         = fitz.open(path)
    total_pages = len(doc)
    documents   = []

    for i, page in enumerate(doc):
        blocks = page.get_text("blocks")

        # ── Collect and combine all blocks ONCE per page ──────
        block_texts = [
            clean_text(block[4])
            for block in blocks
            if len(clean_text(block[4])) >= 20
        ]

        page_text = " ".join(block_texts)

        if not page_text:
            continue

        # ── Append once per page ──────────────────────────────
        documents.append(
            Document(
                page_content = page_text,
                metadata     = {
                    "page"        : i + 1,
                    "total_pages" : total_pages,
                    "source"      : path
                }
            )
        )

    print(f"Loaded {total_pages} pages → {len(documents)} documents")
    return documents

In [ ]:
from langchain.schema import Document
import fitz
import os
import pickle
from langchain.text_splitter import RecursiveCharacterTextSplitter
import tiktoken

# Path to store/load chunks
chunks_file_path = "/content/drive/MyDrive/Colab Notebooks/Medical Assistant/"
path_to_pdf ="/content/drive/MyDrive/Colab Notebooks/Medical Assistant/medical_diagnosis_manual.pdf"

def split_chunks(path_to_pdf, size=512, overlap=50):
  file_name = f"chunks-{size}.{overlap}.pkl"
  chunks_file = os.path.join(chunks_file_path, file_name)
  ## if already processed load chunks from google drive folder
  if (os.path.exists(chunks_file)):
    with open(chunks_file, "rb") as f:
        all_chunks = pickle.load(f)
  else:
  # loading data into a pandas dataframe
    documents = load_pdf_with_metadata(path_to_pdf)
    # Load the cl100k_base tokenizer to ensure chunks are measured by token count rather than character count, which is essential for adhering to LLM context limits
    text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        encoding_name='cl100k_base',
        chunk_size=size,
        chunk_overlap=overlap
    )

    all_chunks = []
    for doc in documents:
        page_text = doc.page_content

        if len(page_text) < 20:
            continue

        chunks = text_splitter.split_text(page_text)
        for j, chunk in enumerate(chunks):
            chunk = clean_text(chunk)
            if chunk:
                all_chunks.append({
                    "text"    : chunk,
                    "metadata": doc.metadata
                })

    with open(chunks_file, "wb") as f:
        pickle.dump(all_chunks, f)
    print(f"Saved {len(all_chunks)} chunks to {file_name}")

  return all_chunks, file_name

Medical content contains abbreviations (e.g., ARDS, TBI), dosage formats (mg/kg, IV, q6h), special characters (%, °C, +, -, /), capitalized disease names, hyphenated terms (such as life-threatening, beta-blocker) Removing or altering these can: Break dosage meaning, Change drug names, Alter diagnosis phrases, Reduce embedding accuracy. We will preserve semantic integrity, hence:

- No stopword removal
- No stemming


#### Chucking the content
Since Content is unstructured with titles, headings, sections, paragraphs of various sizes. Chucking text is the best approach.

In [ ]:
all_chunks, file_name = split_chunks(path_to_pdf);
len(all_chunks)

8125

In [ ]:
print(textwrap.fill(all_chunks[100]["text"]))

Stool cultures should be obtained and checked for ova and parasites if
diarrhea is severe or does not resolve with treatment. Sometimes
urinalysis, urine culture, blood cultures, tuberculin testing, and a
chest x-ray are used to diagnose occult infections because people with
PEU may have a muted response to infections. Children: In children,
mortality varies from 5 to 40%. Mortality rates are lower in children
with mild PEU and those given intensive care. Death in the first days
of treatment is usually due to electrolyte deficits, sepsis,
hypothermia, or heart failure. Impaired consciousness, jaundice,
petechiae, hyponatremia, and persistent diarrhea are ominous signs.
Resolution of apathy, edema, and anorexia is a favorable sign.
Recovery is more rapid in kwashiorkor than in marasmus. Long-term
effects of PEU in children are not fully documented. Some children
develop chronic malabsorption and pancreatic insufficiency. In very
young children, mild intellectual disability may develop a

### **Choice of Embedding Models**

Using the embedding leaderboard, few candidate open-source models selected.
Considering constraints on available compute and memeory (T4: 16 GB GPU RAM)
and considering python packages memory may use over 3 GB, plus chunks)
and the best vocabulary for our medical use case, following three T4 friendly models were compared.

| Model | Size / VRAM Requirement | Type | Strengths | Limitations |
|-------|------------------------|--------|-----------|-------------|
| `BAAI/bge-large-en-v1.5` | Large (~3–4 GB VRAM per batch) | General-purpose | Excellent semantic retrieval,  very high-quality embeddings | Large, may need batching; general-domain, not medical-specific |
| `pritamdeka/S-PubMedBert-MS-MARCO` | Medium (~2 GB VRAM per batch) | Biomedical / PubMed | Domain-specific;better for clinical terminology | Slightly slower than lightweight models;  still needs batching for large corpora |
| `all-mpnet-base-v2` | Medium (~1.5–2 GB VRAM per batch) | General-purpose | Lightweight, fast,  high-quality semantic embeddings | Not medical-specific; may miss subtle clinical terms |



We will use https://huggingface.co/sentence-transformers/all-mpnet-base-v2 model. This is a sentence-transformers model. It maps sentences & paragraphs to a 768 dimensional dense vector space and can be used for tasks like clustering or semantic search. It uses pretrained microsoft/mpnet-base model.

Even though it's not medical-trained, it has strong Semantic Compression. It maps sentences with similar meaning close together, even if wording differs. It generalizes well, has strong semantic structure and works reliably across domains. It is also good for general QA, is fast, stable and handles medical terms well because of semantic compression.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

try:
    embedding_model
except NameError:
    embedding_model = SentenceTransformer("all-mpnet-base-v2")

def generate_embeddings(all_chunks):
  chunk_texts = [chunk["text"] for chunk in all_chunks]
  embeddings = embedding_model.encode(chunk_texts, normalize_embeddings=True, show_progress_bar=True)
  return embeddings

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### **Semantic Search**

Embeddings enable to semantic search for given query using cosine similarity. While below code is there to illustrate this, this search will be enabled using Vector databases to retrieve the context for given query.

In [ ]:
# defining a function to find the top k similar sentences for a given query
def top_k_similar_sentences(embedding_matrix,query_text,k=20):
    # encoding the query text
    query_embedding = embedding_model.encode(query_text, normalize_embeddings=True)

    # calculating the cosine similarity between the query vector and all other encoded vectors of our dataset
    score_vector = np.dot(embedding_matrix,query_embedding)

    # sorting the scores in descending order and choosing the first k
    top_k_indices = np.argsort(score_vector)[::-1][:k]

    # returning the corresponding reviews
    top_values = [all_chunks[i] for i in top_k_indices]
    return top_values

In [ ]:
# Generate vector embeddings for chunked text from pdf using the embedding model we choose earlier
# These embeddings along with chunks will be stored in Chroma and used to supply context for queries
# FAISS as in memory vector index store is only shown here for alternate high performance use cases
try:
    embeddings
except NameError:
    embeddings = generate_embeddings(all_chunks)

Batches:   0%|          | 0/254 [00:00<?, ?it/s]

In [ ]:
top_values = top_k_similar_sentences(embeddings,Query1, 5)
print(top_values)

[{'text': "Parenteral antibiotics should be given after specimens of blood, body fluids, and wound sites have been taken for Gram stain and culture. Very prompt empiric therapy, started immediately after suspecting sepsis, is essential and may be lifesaving. Antibiotic selection requires an educated guess based on the suspected source, clinical setting, knowledge or suspicion of causative organisms and of sensitivity patterns common to that specific inpatient unit, and previous culture results. One regimen for septic shock of unknown cause is gentamicin or tobramycin 5.1 mg/kg IV once/day plus a 3rd-generation cephalosporin (cefotaxime 2 g q 6 to 8 h or ceftriaxone 2 g once/day or, if Pseudomonas is suspected, ceftazidime 2 g IV q 8 h). Alternatively, ceftazidime plus a fluoroquinolone (eg, ciprofloxacin) may be used. Monotherapy with maximal therapeutic doses of ceftazidime (2 g IV q 8 h) or imipenem (1 g IV q 6 h) may be effective but is not recommended. Vancomycin must be added if r

### **Decision Decision: Vector Database**

**FAISS** is a vector similarity search library that stores vectors in the memory. it is extremely fast and has GPU support.  I started with FAISS but then FAISS can only store indexes. Its performance advatange was not a key consideration for our use case because its scale does not require that much performance. I shifted to **ChromaDB**. The code for FAISS stays for illustration but Chorma is the one that is finally used.


In [ ]:
def save_embeddings_index_in_faiss(embeddings, file_name):
  faiss_index_path = "/content/drive/MyDrive/Colab Notebooks/Medical Assistant/"
  faiss_index_file = file_name.replace(".pkl", "") + "_faiss_index.bin"
  faiss_index_file_path = faiss_index_path + faiss_index_file;
  ## if already processed load chunks from google drive folder
  if (os.path.exists(faiss_index_file_path)):
    index = faiss.read_index(faiss_index_file_path)
  else:
    # 3️⃣ Extract text for embedding
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(np.array(embeddings))
    faiss.write_index(index, faiss_index_file_path)
  return index

In [ ]:
def faiss_retrieve(chunks, query, k=20):
    query_vec = embedding_model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(query_vec, k)
    return [all_chunks[i] for i in indices[0]]

In [ ]:
## Save the vector indexes for given embeddings
index = save_embeddings_index_in_faiss(embeddings, file_name)

In [ ]:
print(faiss_retrieve(all_chunks, Query1))

[{'text': "Parenteral antibiotics should be given after specimens of blood, body fluids, and wound sites have been taken for Gram stain and culture. Very prompt empiric therapy, started immediately after suspecting sepsis, is essential and may be lifesaving. Antibiotic selection requires an educated guess based on the suspected source, clinical setting, knowledge or suspicion of causative organisms and of sensitivity patterns common to that specific inpatient unit, and previous culture results. One regimen for septic shock of unknown cause is gentamicin or tobramycin 5.1 mg/kg IV once/day plus a 3rd-generation cephalosporin (cefotaxime 2 g q 6 to 8 h or ceftriaxone 2 g once/day or, if Pseudomonas is suspected, ceftazidime 2 g IV q 8 h). Alternatively, ceftazidime plus a fluoroquinolone (eg, ciprofloxacin) may be used. Monotherapy with maximal therapeutic doses of ceftazidime (2 g IV q 8 h) or imipenem (1 g IV q 6 h) may be effective but is not recommended. Vancomycin must be added if r

**Chorma DB** use is also shown, primarily because it provides persisted storage that is right approach for production systems.
It can store embeddings, metadata, documents ,supports filtering and has built-in persistence.

In [ ]:
import chromadb

chroma_path = "/content/drive/MyDrive/Colab Notebooks/Medical Assistant/chroma_db"

def get_or_create_collection(all_chunks, chunk_size=512, overlap=32):
    collection_name = f"medical_assistant-{chunk_size}.{overlap}"
    client     = chromadb.PersistentClient(path=chroma_path)
    collection = client.get_or_create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"}   # cosine similarity
    )

    # ── Only embed and add if collection is empty ─────────────
    if collection.count() == 0:
        print(f"Embedding and storing {len(all_chunks)} chunks...")
        texts     = [chunk["text"]     for chunk in all_chunks]
        metadatas = [chunk["metadata"] for chunk in all_chunks]
        ids       = [str(i)            for i in range(len(all_chunks))]

        embeddings = embedding_model.encode(
            texts,
            normalize_embeddings=True,
            show_progress_bar=True,
            batch_size=64
        ).tolist()

        # ── Add in batches of 5000 ────────────────────────────
        batch_size = 5000                              # safely under 5461 limit
        total      = len(all_chunks)

        for start in range(0, total, batch_size):
            end = min(start + batch_size, total)       # handle last batch

            collection.add(
                documents  = texts[start:end],
                embeddings = embeddings[start:end],
                metadatas  = metadatas[start:end],
                ids        = ids[start:end]
            )
            print(f"Added chunks {start} - {end} / {total}")

        print(f"Done — stored {collection.count()} chunks in ChromaDB")

    else:
        print(f"Loaded existing collection with {collection.count()} chunks")

    return collection

In [ ]:

def chroma_retrieve(query, collection, k=20):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).tolist()

    results = collection.query(
        query_embeddings = query_embedding,
        n_results        = k,
        include          = ["documents", "metadatas", "distances"]
    )

    return [
        {
            "text":     doc,
            "metadata": meta,
            "score":    1 - dist        # convert distance to similarity score
        }
        for doc, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0]
        )
    ]

In [ ]:
collection = get_or_create_collection(all_chunks);

Loaded existing collection with 8125 chunks


## Re Ranking

Now we have the probable relevant top values, we need to reorder the top retrieved documents based on deeper semantic relevance. We will use a **cross-encoder** to reorder search results to help improve clinical precision.
We will pass the cross-encoder re-ranking model, query + document pairs together as input and outputs a single relevance score.

Re-ranking improves precision context relevance downstream generation quality reduction of hallucinations

### Model selection for Re-ranker

#### Re-Ranking Model Comparison for Medical RAG (Colab T4 Friendly)

| Model | Architecture | Size (Params) | Inference Speed (T4) | Precision | Domain Focus | Pros | Cons | Best Use Case |
|-------|--------------|----------------|----------------------|-----------|---------------|------|------|---------------|
| `cross-encoder/ms-marco-MiniLM-L-6-v2` | Cross-encoder (MiniLM) | ~22M | 🔥 Very Fast | 👍 Good | General | Lightweight, fast, easy | Not medical-trained | Default ranker for general QA/RAG |
| `cross-encoder/ms-marco-MiniLM-L-12-v2` | Cross-encoder (MiniLM) | ~55M | 🔥 Fast | 👍 Better than 6-v2 | General | Better ranking than 6-v2 | Still general domain | Better general domain ranking |
| `cross-encoder/stsb-roberta-base` | Cross-encoder (RoBERTa) | ~125M | ⚡ Moderate | 👍 Solid | General | Strong semantic match | Larger, slower | Semantic similarity focused tasks |
| `hybridsearch-crossencoder-msmarco`* | Cross-encoder specialized | ~110M | ⚡ Moderate | 👍 Higher precision | Retrieval/Re-ranking hybrid | Better end-to-end accuracy | More expensive | RAG pipelines needing high precision |
| `Toxicity-CrossEncoder`** | Various architectures | Varies | ⚡ Moderate | 🛡️ Specialized | Safety/Moderation | Good for safety filtering | Not relevance ranking | Safety/scoring tasks |
| `BAAI/bge-reranker-large` | Cross-encoder (BGE-based) | ~350M+ | 🐢 Slower | ⭐ Very High | General | High quality semantic relevance | Larger, slower inference | When ranking quality > speed |

ms-marco-MiniLM-L-12-v2



In [ ]:
from sentence_transformers import CrossEncoder
try:
  reranker
  print("reranker already loaded")
except NameError:
  reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-12-v2")

reranker already loaded


In [ ]:
def rerank(query, relevant_document_chunks):
    pairs = [[query, chunk["text"]] for chunk in relevant_document_chunks]
    scores = reranker.predict(pairs)
    ranked_idx = np.argsort(scores)[::-1]
    return [relevant_document_chunks[i] for i in ranked_idx[:5]]

## **Generation**

### **Design Decision: Choosing Generation Model (Decoder)**

In the generation part of Retrieval-Augmented Generation (RAG), an LLM (Large Language Model) is needed because it effectively understands complex queries and generates coherent, context-aware responses in natural language. It seamlessly integrates and synthesizes the retrieved context with the user's query, ensuring accurate and relevant answers. LLMs handle ambiguous or complex questions by reasoning over the retrieved knowledge, generating fluent and human-like text for a better user experience.



| Model | Parameters | Domain Focus | Strengths | Limitations | Best Use Case |
|-------|------------|--------------|-----------|-------------|---------------|
| `epfl-llm/meditron-7b` | ~7B | Medical-adapted | Pretrained on medical corpora (PubMed + clinical guidelines), better on medical QA tasks than base LLaMA | Not instruction-tuned by default; may require fine-tuning or instruction wrapping; advisory warns against direct clinical deployment without alignment :contentReference[oaicite:0]{index=0} | Medical reasoning and domain-specific knowledge when combined with instruction tuning |
| `mistralai/Mistral-7B-Instruct-v0.2` | ~7B | General-purpose instruction | Strong performance per parameter vs similar models; efficient and capable with long context workflows; good conversational abilities :contentReference[oaicite:1]{index=1} | Not specialized for medical domain; lacks built-in safety/moderation; broad rather than deep clinical knowledge | Fast general assistant, good for broader tasks and interactive QA |
| `meta-llama/Meta-Llama-3-8B-Instruct` | ~8B | General instruction | Larger model and newer architecture with instruction fine-tuning; robust general reasoning and dialogue :contentReference[oaicite:2]{index=2} | Still general-purpose; moderate context length compared to some others; may require more resources | General RAG generation with stronger reasoning than typical 7B models |
| `TheBloke/Llama-2-13B-chat-GGUF` | ~13B | Chat optimized | Larger size gives stronger knowledge and coherent chat outputs; fine-tuned for dialogue :contentReference[oaicite:3]{index=3} | Larger and slower than 7B models; not domain-specific; requires more memory/cpu/gpu | Highest-quality open-source chat for broad contexts if resources allow |



We will choose meta-llama/Meta-Llama-3-8B-Instruct which is same as the model we initially started with ("bartowski/Meta-Llama-3-8B-Instruct-GGUF") just in a different format.

The model is selected based on:

* Very strong reasoning ability

* Excellent instruction following

* Handles complex explanations well

Model is suppose to perform well without medical specialization as
medical knowledge exists in general training data, reasoning quality compensates for lack of specialization
and it works very well with RAG and prompt engineering.

### **Design Decision: Context Window**

Llama allows context window of 8192 tokens. We need max 5000 tokens for k=3 to 5 for chunk size of 512

(num_chunks × chunk_size) + system_prompt + question + max_new_tokens < context_window

In [ ]:
llm = Llama(
    model_path=bartowski_model_path,
    n_ctx=5000,
    n_gpu_layers=38,
    n_batch=512
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


# Question Answering using RAG

### System and User Prompt Template

In [ ]:
qna_system_message = """<|start_header_id|>system<|end_header_id|>
You are a medical diagnosis assistant that answers strictly using Merck Manual content.

Rules:
- Answer ONLY using the provided context. Do not use outside knowledge.
- If the answer cannot be derived from the Context, respond with "I don't know".
- The answer may not always be directly stated — use reasoning within the context.

Response format:
- Clinical Explanation
- Treatment Protocol
- Cite section titles if available
- Use bullet points throughout
<|eot_id|>
"""

In [ ]:
qna_user_message_template = """<|start_header_id|>user<|end_header_id|>
<context>
#context
</context>
<question>
#question
</question>
<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

### Response Function

In [ ]:
def create_prompt(relevant_document_chunks, system_prompt, user_prompt_template, user_input):

    context_for_query = ". ".join([chunk["text"] for chunk in relevant_document_chunks])

    user_message = user_prompt_template.replace('#context', context_for_query)
    user_message = user_message.replace('#question', user_input)

    prompt = system_prompt + '\n' + user_message

    return prompt

In [ ]:
def generate_rag_response(prompt,max_tokens=256,temperature=0,top_p=0.95,top_k=10):
    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k,
                  stop        = ["<|eot_id|>",
                           "<|start_header_id|>",  "\nassistant", "\nuser"]
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
relevant_document_chunks = chroma_retrieve(Query1, collection)
pretty_print_doc_collection(relevant_document_chunks)
reranked_chunks = rerank(Query1, relevant_document_chunks)

── Chunk 1 ──────────────────────────────────────────
Page   : 2457
Score  : 0.6607
Text   :
Parenteral antibiotics should be given after specimens of blood, body fluids,
and wound sites have been taken for Gram stain and culture. Very prompt empiric
therapy, started immediately after suspecting sepsis, is essential and may be
lifesaving. Antibiotic selection requires an educated guess based on the
suspected source, clinical setting, knowledge or suspicion of causative
organisms and of sensitivity patterns common to that specific inpatient unit,
and previous culture results. One regimen for septic shock of unknown cause is
gentamicin or tobramycin 5.1 mg/kg IV once/day plus a 3rd-generation
cephalosporin (cefotaxime 2 g q 6 to 8 h or ceftriaxone 2 g once/day or, if
Pseudomonas is suspected, ceftazidime 2 g IV q 8 h). Alternatively, ceftazidime
plus a fluoroquinolone (eg, ciprofloxacin) may be used. Monotherapy with maximal
therapeutic doses of ceftazidime (2 g IV q 8 h) or imipenem (1 

In [ ]:
prompt = create_prompt(reranked_chunks,qna_system_message,qna_user_message_template, Query1)
print(prompt)
print(generate_rag_response(prompt));

<|start_header_id|>system<|end_header_id|>
You are a medical diagnosis assistant that answers strictly using Merck Manual content.

Rules:
- Answer ONLY using the provided context. Do not use outside knowledge.
- If the answer cannot be derived from the Context, respond with "I don't know".
- The answer may not always be directly stated — use reasoning within the context.

Response format:
- Clinical Explanation
- Treatment Protocol
- Cite section titles if available
- Use bullet points throughout
<|eot_id|>

<|start_header_id|>user<|end_header_id|>
<context>
16 - Critical Care Medicine Chapter 222. Approach to the Critically Ill Patient Critical care medicine specializes in caring for the most seriously ill patients. These patients are best treated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special populations (eg, cardiac, surgical, neurologic, pediatric, or neonatal patients). ICUs have a high nurse:patient ratio to provide the necessary h

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
relevant_document_chunks = chroma_retrieve(Query2, collection)
#pretty_print_doc_collection(relevant_document_chunks)
reranked_chunks = rerank(Query2, relevant_document_chunks)

In [ ]:
prompt = create_prompt(reranked_chunks,qna_system_message,qna_user_message_template, Query2)
print(prompt)
print(generate_rag_response(prompt));

<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a medical diagnosis assistant that answers strictly using Merck Manual content.

Rules:
- Answer ONLY using the provided context. Do not use outside knowledge.
- If the answer cannot be derived from the Context, respond with "I don't know".
- The answer may not always be directly stated — use reasoning within the context.

Response format:
- Clinical Explanation
- Treatment Protocol
- Cite section titles if available
- Use bullet points throughout
<|eot_id|>

<|start_header_id|>user<|end_header_id|>
<context>
without delay. Contrastenhanced CT has reasonable accuracy in diagnosing appendicitis and can also reveal other causes of an acute abdomen. Graded compression ultrasound can usually be done quickly and uses no radiation (of particular concern in children); however, it is occasionally limited by the presence of bowel gas and is less useful for recognizing nonappendiceal causes of pain. Appendicitis remains primaril

Llama.generate: prefix-match hit


**Clinical Explanation**

Appendicitis is characterized by sudden onset of abdominal pain, anorexia, and abdominal tenderness. The diagnosis is primarily clinical, often supplemented by CT or ultrasound.

**Treatment Protocol**

* Treatment of acute appendicitis is open or laparoscopic appendectomy.
* IV fluids and antibiotics are administered before surgery.
* Third-generation cephalosporins are preferred for antibiotic treatment.
* For non-perforated appendicitis, no further antibiotics are required after initial treatment.
* If the appendix is perforated, antibiotics should be continued until the patient's temperature and WBC count have normalized or continued for a fixed course, according to the surgeon's preference.

**Cite**

* The Merck Manual of Diagnosis & Therapy, 19th Edition Chapter 11. Acute Abdomen & Surgical Gastroenterology

**Note**

Appendicitis cannot be cured via medicine alone. Surgery is necessary to remove the inflamed appendix. Delayed treatment increases mortal

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
relevant_document_chunks = chroma_retrieve(Query3, collection)
#pretty_print_doc_collection(relevant_document_chunks)
reranked_chunks = rerank(Query3, relevant_document_chunks)

In [ ]:
prompt = create_prompt(reranked_chunks,qna_system_message,qna_user_message_template, Query3)
print(prompt)
print(generate_rag_response(prompt));

<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a medical diagnosis assistant that answers strictly using Merck Manual content.

Rules:
- Answer ONLY using the provided context. Do not use outside knowledge.
- If the answer cannot be derived from the Context, respond with "I don't know".
- The answer may not always be directly stated — use reasoning within the context.

Response format:
- Clinical Explanation
- Treatment Protocol
- Cite section titles if available
- Use bullet points throughout
<|eot_id|>

<|start_header_id|>user<|end_header_id|>
<context>
with oral antimalarials, corticosteroids, retinoids, or immunosuppressants. Hair loss due to chemotherapy is temporary and is best treated with a wig; when hair regrows, it may be different in color and texture from the original hair. Hair loss due to telogen effluvium or anagen effluvium is usually temporary as well and abates after the precipitating agent is eliminated. • Androgenetic alopecia (male-pattern and 

Llama.generate: prefix-match hit


**Clinical Explanation**

Sudden patchy hair loss, also known as alopecia areata, is a common condition characterized by the sudden onset of patchy hair loss on the scalp or other hairy areas of the body. It is thought to be an autoimmune disorder that affects genetically susceptible individuals exposed to unclear environmental triggers.

**Treatment Protocol**

The treatment for alopecia areata depends on the extent and severity of the hair loss. In most cases, it is a self-limiting condition, meaning it will resolve on its own without treatment. However, there are several treatments available to promote hair regrowth and reduce the appearance of bald patches:

* Topical corticosteroids: applied directly to the affected area to reduce inflammation
* Minoxidil (Rogaine): a topical solution that stimulates hair growth and slows down hair loss
* Finasteride (Propecia): an oral medication that slows down hair loss and promotes hair regrowth
* Phototherapy: exposure to ultraviolet light or

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
relevant_document_chunks = chroma_retrieve(Query4, collection)
#pretty_print_doc_collection(relevant_document_chunks)
reranked_chunks = rerank(Query4, relevant_document_chunks)

In [ ]:
prompt = create_prompt(reranked_chunks,qna_system_message,qna_user_message_template, Query4)
print(prompt)
print(generate_rag_response(prompt));

<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a medical diagnosis assistant that answers strictly using Merck Manual content.

Rules:
- Answer ONLY using the provided context. Do not use outside knowledge.
- If the answer cannot be derived from the Context, respond with "I don't know".
- The answer may not always be directly stated — use reasoning within the context.

Response format:
- Clinical Explanation
- Treatment Protocol
- Cite section titles if available
- Use bullet points throughout
<|eot_id|>

<|start_header_id|>user<|end_header_id|>
<context>
Chapter 324. Traumatic Brain Injury Traumatic brain injury (TBI) is physical injury to brain tissue that temporarily or permanently impairs brain function. Diagnosis is suspected clinically and confirmed by imaging (primarily CT). Initial treatment consists of ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery is often needed in patients with more severe injur

Llama.generate: prefix-match hit


**Clinical Explanation**

Traumatic brain injury (TBI) requires immediate attention to ensure a reliable airway and adequate ventilation, oxygenation, and blood pressure. Surgery may be necessary to place monitors for intracranial pressure management, decompress the brain if pressure is increased, or remove intracranial hematomas.

**Treatment Protocol**

* Ensure a reliable airway
* Maintain adequate ventilation, oxygenation, and blood pressure
* Consider surgery to:
	+ Place monitors for intracranial pressure management
	+ Decompress the brain if pressure is increased
	+ Remove intracranial hematomas
* Manage hypotension by giving fluids and vasopressors as needed
* Monitor and treat seizures promptly, especially in patients with significant structural injury or a Glasgow Coma Scale (GCS) score < 10
* Consider prophylactic anticonvulsants for patients with significant structural injury or GCS < 10
* Provide therapeutic hypothermia to suppress bursts of EEG activity
* Monitor and mana

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
relevant_document_chunks = chroma_retrieve(Query5, collection)
#pretty_print_doc_collection(relevant_document_chunks)
reranked_chunks = rerank(Query5, relevant_document_chunks)

In [ ]:
prompt = create_prompt(reranked_chunks,qna_system_message,qna_user_message_template, Query5)
print(prompt)
print(generate_rag_response(prompt));

<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a medical diagnosis assistant that answers strictly using Merck Manual content.

Rules:
- Answer ONLY using the provided context. Do not use outside knowledge.
- If the answer cannot be derived from the Context, respond with "I don't know".
- The answer may not always be directly stated — use reasoning within the context.

Response format:
- Clinical Explanation
- Treatment Protocol
- Cite section titles if available
- Use bullet points throughout
<|eot_id|>

<|start_header_id|>user<|end_header_id|>
<context>
from injuring the skin • To seek medical care at once if an odor emanates from within the cast or if a fever, which may indicate Good hygiene is important. A splint (see Fig. 323-1) can be used to immobilize some stable injuries, including some suspected but unproven fractures, sprains, and other injuries that require immobilization for several days or less. A splint allows patients to apply ice and to move more a

Llama.generate: prefix-match hit


**Clinical Explanation**

A fracture is a crack in the bone that can occur due to various mechanisms such as trauma, osteoporosis, or underlying medical conditions. In this case, the person has fractured their leg during a hiking trip. The primary concern is to ensure proper immobilization and management of pain to prevent further injury and promote healing.

**Treatment Protocol**

1. **Immobilization**: A splint can be used to immobilize the injured leg, allowing for rest and reducing the risk of further injury.
2. **Pain Management**: Analgesics may be prescribed to manage pain and discomfort.
3. **Rest**: The person should avoid putting weight on the affected leg and rest it as much as possible.
4. **Ice and Compression**: Ice packs can be applied intermittently for 15-20 minutes, several times a day, to reduce swelling and pain. Compression bandages or splints can also help minimize swelling.
5. **Elevation**: Elevating the injured leg above the level of the heart can help reduce 

### **Observations**

## **Fine-tuning**

Fine tuning the RAG Architecture, involve picking the right model, **optimal chunking**, **optimal retirval** and **generation**. Below are the 3 different combination that we will try with **Deterministic** and **Conservative Hyperparameter** with **Constrainted Prompting** these we will try to achieve optimal results
| Experiment | Chunk Size | Overlap | TopK | Embedding Model      | LLM Model|
|------------|------------|---------|------|----------------------|------------------------|
| 1          | 256        | 32      | 3    | all-mpnet-base-v2    | Llama-3-8B-Instruct    |
| 2          | 512        | 64      | 5    | all-mpnet-base-v2    | Llama-3-8B-Instruct    |
| 3          | 768        | 64      | 5    | all-mpnet-base-v2    | Llama-3-8B-Instruct    |

We will perform above three experiments with **Deterministic** and **Conservative** LLM configuraiton. Since this is Medical Assistant to provide tools to support quick decision-making and enhance efficiency, we will experiment with towards Deterministic settings.


| Strategy | temperature | top_p | top_k | max_tokens | Use Case |
|---|---|---|---|---|---|
| Deterministic | 0 | 0.95 | 10 | 256 | Factual, consistent answers |
| Conservative | 0.1 | 0.9 | 20 | 256 | Slight variation, still safe |


## **Configuration 1**


This is base case for **Deterministic behavior**. Max output token is 256 which can be a challenge as it potentially can truncate response.

In [ ]:
# Chunk PDF Manual
all_chunks_256_32, file_name = split_chunks(path_to_pdf, 256, 32)
collection = get_or_create_collection(all_chunks_256_32, 256,32);
relevant_document_chunks = chroma_retrieve(Query1, collection)
reranked_chunks = rerank(Query1, relevant_document_chunks)
prompt = create_prompt(reranked_chunks,qna_system_message,qna_user_message_template, Query1)
response = generate_rag_response(prompt) # k=3,max_tokens=256,temperature=0,top_p=0.95,top_k=10):
print(response)

Loaded existing collection with 8125 chunks
**Clinical Explanation**

Sepsis is a life-threatening condition that requires prompt recognition and management. In the critical care unit, the goal of treatment is to identify and eliminate the source of infection, provide supportive care, and manage organ dysfunction.

**Treatment Protocol**

1. **Initial Assessment**: Rapidly assess the patient's condition, including vital signs, laboratory results, and physical examination.
2. **Empiric Antibiotic Therapy**: Administer broad-spectrum antibiotics based on the suspected source of infection, clinical setting, and knowledge of common pathogens in the unit.
3. **Fluid Resuscitation**: Provide aggressive fluid resuscitation to maintain adequate blood pressure and perfusion.
4. **Supportive Care**: Manage organ dysfunction by providing mechanical ventilation, renal replacement therapy, and other supportive measures as needed.
5. **Source Control**: Eliminate the source of infection by draining 

## **Configuration 2**

* Notice the temperature=0.1,
* top_p=0.9.
* Chunck and overlap stays same.
* Reranking model and configuration stays same.

In [ ]:
relevant_document_chunks = chroma_retrieve(Query1, collection)
reranked_chunks = rerank(Query1, relevant_document_chunks)
prompt = create_prompt(reranked_chunks,qna_system_message,qna_user_message_template, Query1)
response = generate_rag_response(prompt, k=3,max_tokens=256,temperature=0.1,top_p=0.9,top_k=20)
print(response)

Loaded existing collection with 8125 chunks


Llama.generate: prefix-match hit


**Clinical Explanation**

Sepsis is a life-threatening condition that requires prompt recognition and management. In the critical care unit, the goal is to identify early signs of sepsis, initiate empiric antibiotic therapy, and provide supportive care to prevent organ dysfunction.

**Treatment Protocol**

1. **Initial Assessment**: Monitor vital signs, including temperature, blood pressure, pulse, and respiratory rate.
2. **Empiric Antibiotic Therapy**: Administer broad-spectrum antibiotics based on the suspected source of infection, clinical setting, and knowledge of common causative organisms.
3. **Fluid Resuscitation**: Provide aggressive fluid resuscitation to maintain adequate perfusion and prevent organ dysfunction.
4. **Supportive Care**:
	* Monitor and manage blood glucose levels to prevent hyperglycemia or hypoglycemia.
	* Administer corticosteroids for septic shock and acute respiratory distress syndrome (ARDS).
	* Provide mechanical ventilation as needed for respiratory fai

## **Configuration 3**

* Notice the shift to Deterministic setting temperature=0,top_p=0.95.
* Chunks and Overlap increases to 768 and 64

In [ ]:
all_chunks_768_64, file_name = split_chunks(path_to_pdf, 768, 64)
collection = get_or_create_collection(all_chunks_768_64,768, 64);
relevant_document_chunks = chroma_retrieve(Query1, collection)
reranked_chunks = rerank(Query1, relevant_document_chunks)
prompt = create_prompt(reranked_chunks,qna_system_message,qna_user_message_template, Query1)
response = generate_rag_response(prompt) # k=3,max_tokens=256,temperature=0,top_p=0.95,top_k=10):
print(response)

Loaded existing collection with 8125 chunks


Llama.generate: prefix-match hit


**Clinical Explanation**

Sepsis is a life-threatening condition that requires prompt recognition and management. In the critical care unit, the goal is to identify early signs of sepsis, initiate empiric antibiotic therapy, and provide supportive care to prevent organ dysfunction.

**Treatment Protocol**

1. **Initial Assessment**: Monitor vital signs, including temperature, blood pressure, pulse, and respiratory rate.
2. **Empiric Antibiotic Therapy**: Administer broad-spectrum antibiotics based on the suspected source of infection, clinical setting, and knowledge of common causative organisms.
3. **Fluid Resuscitation**: Provide aggressive fluid resuscitation to maintain adequate perfusion and prevent organ dysfunction.
4. **Supportive Care**:
	* Monitor and manage blood glucose levels to prevent hyperglycemia or hypoglycemia.
	* Administer corticosteroids for septic shock, if necessary.
	* Consider activated protein C therapy in patients with severe sepsis and septic shock.
5. **So

## **Configuration 4**

* Notice the shift to Conservative setting temperature=0.1,top_p=0.9
* Chunks and Overlap stays same to 768 and 64
* Top K stays at 10
* Top N (After Reranking) stays at 3
* Lower number compensate for higher token size of 512

In [ ]:
## Using same collection for chunks_size=768, overlap=64
relevant_document_chunks = chroma_retrieve(Query1, collection)
reranked_chunks = rerank(Query1, relevant_document_chunks)
prompt = create_prompt(reranked_chunks,qna_system_message,qna_user_message_template, Query1)
response = generate_rag_response(prompt,max_tokens=512,temperature=0.1,top_p=0.9,top_k=20)
print(response)

Loaded existing collection with 8125 chunks


Llama.generate: prefix-match hit


**Clinical Explanation**

Sepsis is a life-threatening condition that requires prompt recognition and management. In the critical care unit, the goal of treatment is to identify and eliminate the source of infection, provide supportive care, and manage organ dysfunction.

**Treatment Protocol**

1. **Initial Assessment**: Rapidly assess the patient's clinical status, including vital signs, laboratory results, and imaging studies.
2. **Antibiotic Therapy**: Administer empiric antibiotics based on the suspected source of infection, clinical presentation, and local antibiotic resistance patterns.
3. **Fluid Resuscitation**: Provide aggressive fluid resuscitation with crystalloid or colloid solutions to maintain adequate blood pressure and perfusion.
4. **Supportive Care**: Manage organ dysfunction by providing mechanical ventilation, vasopressors, and other supportive measures as needed.
5. **Source Control**: Eliminate the source of infection by draining abscesses, removing infected devi

## **Configuration 5**

* Chunk size: 512
* Overlap: 64
* Embedding model: all-mpnet-base-v2
* Vector DB: ChromaDB
* LLM: Llama-3-8B-Instruct (GGUF)
* Temperature: 0
* Max tokens: 512
* Prompt: grounded medical system prompt

In [ ]:
all_chunks_512_64, file_name = split_chunks(path_to_pdf, 512, 64)
collection = get_or_create_collection(all_chunks_512_64, 512,64);
relevant_document_chunks = chroma_retrieve(Query1, collection)
reranked_chunks1 = rerank(Query1, relevant_document_chunks)
prompt = create_prompt(reranked_chunks1,qna_system_message,qna_user_message_template, Query1)
Query1_response = generate_rag_response(prompt, max_tokens=512,temperature=0,top_p=0.95,top_k=3)
print(Query1_response)

Loaded existing collection with 8200 chunks


Llama.generate: prefix-match hit


**Clinical Explanation**

Sepsis is a life-threatening condition that requires prompt recognition and management. In a critical care unit, the protocol for managing sepsis involves a multi-faceted approach that includes:

* Aggressive fluid resuscitation with 0.9% normal saline to maintain adequate blood pressure and perfusion
* Broad-spectrum antibiotics administered empirically based on suspected source of infection and clinical setting
* Drainage of abscesses and excision of necrotic tissue to eliminate septic foci
* Normalization of blood glucose levels through continuous IV insulin infusion
* Replacement-dose corticosteroids to support adrenal function

**Treatment Protocol**

1. **Initial Assessment**: Rapidly assess the patient's condition, including vital signs, laboratory results, and clinical presentation.
2. **Fluid Resuscitation**: Administer 0.9% normal saline at a rate of 500-1000 mL/h to maintain adequate blood pressure and perfusion.
3. **Antibiotic Therapy**: Initiate 

| Configuration | Name | Chunk Size | Overlap | Temperature | top_p | top_k | max_tokens | Response Quality | Issues |
|---|---|---|---|---|---|---|---|---|---|
| 1 | Small Chunks Deterministic | 256 | 32 | 0 | 0.95 | 10 | 256 | Good structure, covers key protocol steps | Cut off mid-answer, small chunks miss context |
| 2 | Small Chunks Conservative | 256 | 32 | 0.1 | 0.90 | 20 | 256 | More detailed, adds corticosteroids and vasopressors | Cut off mid-sentence |
| 3 | Large Chunks Deterministic | 768 | 64 | 0 | 0.95 | 10 | 256 | Adds activated protein C therapy | Cut off mid-sentence, outdated treatment (withdrawn 2011) |
| 4 | Large Chunks Conservative | 768 | 64 | 0.1 | 0.90 | 20 | 512 | Most complete — cites chapters, blood glucose targets | Stop token bug causes self-evaluation after answer |
| 5 | Medium Chunks Deterministic | 512 | 64 | 0 | 0.95 | 5 | 512 | Balanced chunk size and deterministic generation optimized for accuracy | Best trade-off between context quality, retrieval precision, and response length. |


In [ ]:
# Query2 Response
relevant_document_chunks = chroma_retrieve(Query2, collection)
reranked_chunks2 = rerank(Query2, relevant_document_chunks)
prompt = create_prompt(reranked_chunks2,qna_system_message,qna_user_message_template, Query2)
Query2_response = generate_rag_response(prompt, max_tokens=512,temperature=0,top_p=0.95,top_k=5)
print(Query2_response)

Llama.generate: prefix-match hit


**Clinical Explanation**

Appendicitis is characterized by acute inflammation of the vermiform appendix, typically resulting in abdominal pain, anorexia, and abdominal tenderness. The classic symptoms of acute appendicitis are epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia; after a few hours, the pain shifts to the right lower quadrant. Additional signs include right lower quadrant direct and rebound tenderness located at McBurney's point, Rovsing sign, psoas sign, and obturator sign.

**Treatment Protocol**

Appendicitis is typically treated with surgical removal of the appendix, either through open or laparoscopic appendectomy. The surgeon can usually remove the appendix even if perforated. IV fluids and antibiotics are administered preoperatively to help manage symptoms and prevent complications.

For nonperforated appendicitis, no further antibiotics are required after surgery. If the appendix is perforated, antibiotics should be continued until t

In [ ]:
# Query3 Response
relevant_document_chunks = chroma_retrieve(Query3, collection)
reranked_chunks3 = rerank(Query3, relevant_document_chunks)
prompt = create_prompt(reranked_chunks3,qna_system_message,qna_user_message_template, Query3)
Query3_response = generate_rag_response(prompt, max_tokens=512,temperature=0,top_p=0.95,top_k=5)
print(Query3_response)

Llama.generate: prefix-match hit


**Clinical Explanation**

Sudden patchy hair loss, also known as alopecia areata, is an autoimmune disorder characterized by the sudden onset of patchy hair loss on the scalp or other hairy areas. It can occur at any age but is most common in children and young adults.

The possible causes behind alopecia areata include:

* Genetic predisposition
* Environmental triggers (e.g., stress, infection)
* Hormonal changes
* Autoimmune disorders

**Treatment Protocol**

Effective treatments for alopecia areata include:

1. **Topical corticosteroids**: Triamcinolone acetonide suspension can be injected intradermally or potent topical corticosteroids like betamethasone 0.05% bid can be used.
2. **Minoxidil**: Topical minoxidil 1 mL bid applied to the scalp is most effective for vertex alopecia in male-pattern or female-pattern hair loss.
3. **Anthralin**: Topical anthralin (0.5 to 1% for 10 to 20 min daily, then washed off) can be used.
4. **Immunotherapy**: Induction of allergic contact dermati

In [ ]:
# Query4 Response
relevant_document_chunks = chroma_retrieve(Query4, collection)
reranked_chunks4 = rerank(Query4, relevant_document_chunks)
prompt = create_prompt(reranked_chunks4,qna_system_message,qna_user_message_template, Query4)
Query4_response = generate_rag_response(prompt, max_tokens=512,temperature=0,top_p=0.95,top_k=5)
print(Query4_response)

Llama.generate: prefix-match hit


**Clinical Explanation**

Traumatic brain injury (TBI) is a physical injury to brain tissue that temporarily or permanently impairs brain function. The initial treatment consists of ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed in patients with more severe injuries to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas.

**Treatment Protocol**

* Ensure a reliable airway
* Maintain adequate ventilation, oxygenation, and blood pressure
* Monitor for and manage increased intracranial pressure (ICP)
* Consider surgery in patients with more severe injuries to:
	+ Place monitors to track ICP
	+ Decompress the brain if ICP is increased
	+ Remove intracranial hematomas
* Maintain adequate brain perfusion and oxygenation
* Prevent complications of altered sensorium

**Cite**

Chapter 324. Traumatic Brain Injury, The Merck Ma

In [ ]:
# Query5 Response
relevant_document_chunks = chroma_retrieve(Query5, collection)
reranked_chunks5 = rerank(Query5, relevant_document_chunks)
prompt = create_prompt(reranked_chunks5,qna_system_message,qna_user_message_template, Query5)
Query5_response = generate_rag_response(prompt, max_tokens=512,temperature=0,top_p=0.95,top_k=5)
print(Query5_response)

Llama.generate: prefix-match hit


Clinical Explanation:

A fracture of the leg can occur due to various reasons such as trauma, overuse, or osteoporosis. The symptoms may include pain, swelling, bruising, and limited mobility in the affected limb. It is essential to seek medical attention immediately if you suspect a fracture.

Treatment Protocol:

1. Immobilization: Apply a splint or cast to immobilize the leg and reduce pain.
2. Pain management: Use analgesics such as acetaminophen or NSAIDs to manage pain.
3. Rest: Avoid putting weight on the affected limb and rest it as much as possible.
4. Elevation: Elevate the affected limb above the level of the heart to reduce swelling.
5. Ice application: Apply ice packs to the affected area for 15-20 minutes, several times a day, to reduce pain and inflammation.
6. Compression: Use an elastic bandage or compression wrap to compress the affected area and reduce swelling.
7. Rehabilitation: Gradually increase mobility and strength exercises under the guidance of a healthcare p

# Output Evaluation

We will use the judge model to evaluates:

* Faithfulness – Is the answer supported by retrieved context?
* Relevance – Did it answer the question?
* Medical correctness – Is the treatment aligned with the manual?
* Completeness – Did it miss critical protocol steps?
* Hallucination detection – Did it add unsupported claims?

The right judge model requires strong reasoning, strong instruction following, stable output, not necessarily huge parameter size. It can't be same model as generation model as that will introduce bias. I will use meta-llama/Meta-Llama-3-8B-Instruct because of its stability, performance, strong reasoning and it can perform with Colab T4.

In [ ]:
judge_model_name_or_path="TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
judge_model_basename="mistral-7b-instruct-v0.2.Q4_K_M.gguf"

In [ ]:
judgellm_model_path = hf_hub_download(
    repo_id=judge_model_name_or_path,
    filename=judge_model_basename
)

In [ ]:
#uncomment the below snippet of code if the runtime is connected to GPU.
judge_llm = Llama(
    model_path=judgellm_model_path,
    n_ctx=5000,
    n_gpu_layers=38,
    n_batch=512
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


## Groundedness

Below groundedness rater prompt uses xml tags making them less ambigious. The exmaples (Few-Shot prompt engineering) are provided as reference to model. Model has been told to produce json output in very structured expected format.

We are going to use our knowledge base created from Merck manual to generate an example for prompt.

In [ ]:
groundedness_rater_system_message = """
You are a strict medical QA evaluator.

You will be given a <question>, <context>, and an AI-generated <answer>.

Your task: Judge whether the <answer> is derived from and supported by the <context>.

Scoring rubric:
- Score 1: Answer contradicts the context or introduces facts not present in context
- Score 2: Answer is loosely related to context but makes unsupported claims
- Score 3: Answer uses context but includes some details not found in context
- Score 4: Answer is mostly grounded in context with minor gaps
- Score 5: Every claim in the answer is directly supported by the context

## Examples

<question>What causes Type 2 diabetes mellitus in adolescents?</question>
<context>Type 2 diabetes mellitus is occurring with increasing frequency due to obesity in adolescents</context>
<answer>Type 2 diabetes mellitus is occurring with increasing frequency due to obesity in adolescents</answer>
{"score": 5, "reason": "Claim directly stated in context.", "unsupported_claims": "none"}

---
<question>What causes Type 2 diabetes?</question>
<context>Type 2 diabetes mellitus is occurring with increasing frequency due to obesity in adolescents</context>
<answer>Type 2 diabetes is caused by sedentary lifestyle.</answer>
{"score": 1, "reason": "Sedentary lifestyle not mentioned in context.", "unsupported_claims": "sedentary lifestyle"}

---
IMPORTANT: Provide your answer ONCE. Do not evaluate or comment on your response.
IMPORTANT: Respond ONLY in this exact JSON format, no other text:
{
  "score": <integer 1-5>,
  "reason": "<one sentence explanation>",
  "unsupported_claims": "<list any claims not found in context, or 'none'>"
}
"""

In [ ]:
relevance_rater_system_message = """
You are a strict medical QA evaluator.

You will be given a <question>, <context>, and an AI-generated <answer>.

Your task: Judge whether the <answer> directly and completely addresses the <question>.

Scoring rubric:
- Score 1: Answer is completely off-topic or does not address the question
- Score 2: Answer addresses the topic but misses the core ask of the question
- Score 3: Answer partially addresses the question but omits key aspects
- Score 4: Answer addresses the question well with minor omissions
- Score 5: Answer directly and completely addresses all aspects of the question

## Examples

<question>What are the symptoms of hypertension?</question>
<context>Hypertension often has no symptoms but can cause headaches and vision problems.</context>
<answer>Hypertension can cause headaches and vision problems, and is often asymptomatic.</answer>
{"score": 5, "reason": "Directly answers all aspects of the question.", "missing_aspects": "none"}

---
<question>What are the symptoms of hypertension?</question>
<context>Hypertension often has no symptoms but can cause headaches and vision problems.</context>
<answer>Hypertension is a serious cardiovascular condition affecting millions.</answer>
{"score": 1, "reason": "Does not address symptoms at all.", "missing_aspects": "all symptoms"}

---
<question>What are the symptoms of hypertension?</question>
<context>Hypertension often has no symptoms but can cause headaches and vision problems.</context>
<answer>Hypertension can cause headaches.</answer>
{"score": 3, "reason": "Mentions headaches but omits vision problems and asymptomatic nature.", "missing_aspects": "vision problems, asymptomatic cases"}

IMPORTANT: Provide your answer ONCE. Do not evaluate or comment on your response.
IMPORTANT: Respond ONLY in this exact JSON format, no other text:
{
  "score": <integer 1-5>,
  "reason": "<one sentence explanation>",
  "missing_aspects": "<what key aspects of the question were not addressed, or 'none'>"
}
"""

In [ ]:
faithfulness_rater_system_message = """
You are a strict medical QA evaluator.

You will be given a <question>, <context>, and an AI-generated <answer>.

Your task: Detect whether the <answer> contains hallucinated facts —
claims that are NEITHER supported by the <context> NOR established medical knowledge.

## Examples

<question>What is the treatment for Type 2 diabetes?</question>
<context>Type 2 diabetes is managed with metformin, lifestyle changes, and blood sugar monitoring.</context>
<answer>The primary treatment for type 2 DM is oral antihyperglycemic drugs</answer>
{"is_faithful": true, "hallucinated_claims": "none", "severity": "none"}

---
<question>What is the treatment for Type 2 diabetes?</question>
<context>Type 2 diabetes is managed with metformin, lifestyle changes, and blood sugar monitoring.</context>
<answer>Type 2 diabetes is cured by insulin injections taken three times daily.</answer>
{"is_faithful": false, "hallucinated_claims": "insulin injections three times daily, claimed as cure", "severity": "major"}

---
<question>What is the treatment for Type 2 diabetes?</question>
<context>Type 2 diabetes is managed with metformin, lifestyle changes, and blood sugar monitoring.</context>
<answer>Type 2 diabetes is treated with metformin. Patients should also avoid sugar completely.</answer>
{"is_faithful": false, "hallucinated_claims": "avoid sugar completely", "severity": "minor"}

CRITICAL INSTRUCTIONS:
- Output ONLY one JSON object
- Do NOT change your answer after producing JSON
- Do NOT apologize or revise
- Stop immediately after the closing brace }

...scoring rubric and examples...

IMPORTANT: Respond ONLY in this exact JSON format:
{
  "is_faithful": <true or false>,
  "hallucinated_claims": "<claims or none>",
  "severity": "<none | minor | major>"
}
"""

In [ ]:
# Single template used across all 3 raters
judge_user_message_template = """
<question>{question}</question>
<context>{context}</context>
<answer>{answer}</answer>
"""

In [ ]:
def build_llama3_prompt(system_message, user_message):
    return (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n{system_message}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n{user_message}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>"
    )

In [ ]:
def evaluate_with_judge(judge_llm, question, answer,
                        reranked_chunks, max_tokens=150):
    raters = {
        "groundedness": groundedness_rater_system_message,
        "relevance":    relevance_rater_system_message,
        "faithfulness": faithfulness_rater_system_message,
    }

    context_for_query = ". ".join([chunk["text"] for chunk in reranked_chunks])

    judge_user_message = judge_user_message_template.format(
        question = question,
        context  = context_for_query,
        answer   = answer
    )

    results = {}
    for metric, system_msg in raters.items():
        prompt = build_llama3_prompt(system_msg, judge_user_message)
        raw = judge_llm(
            prompt      = prompt,
            max_tokens  = max_tokens,
            temperature = 0,
            top_p       = 0.95,
            top_k       = 10,
            stop        = [
            "<|eot_id|>",
            "<|start_header_id|>",
            "\nassistant",
            "\nuser",
        ]
        )["choices"][0]["text"].strip()

        # ── Extract first JSON block only ─────────────────────
        try:
            json_start = raw.find("{")
            json_end   = raw.find("}") + 1
            if json_start == -1 or json_end == 0:
                raise ValueError("No JSON found")
            clean = raw[json_start:json_end]
            results[metric] = json.loads(clean)
        except (json.JSONDecodeError, ValueError):
            results[metric] = {"error": "parse_failed", "raw": raw}

    return results

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
evaluate_with_judge(judge_llm,Query1, Query1_response, reranked_chunks1)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


{'groundedness': {'score': 5,
  'reason': 'Answer is directly supported by the context.',
  'unsupported_claims': 'none'},
 'relevance': {'score': 5,
  'reason': 'Directly answers all aspects of the question, providing a comprehensive protocol for managing sepsis in a critical care unit.',
  'missing_aspects': 'none'},
 'faithfulness': {'is_faithful': False,
  'hallucinated_claims': 'none',
  'severity': 'minor'}}

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
evaluate_with_judge(judge_llm,Query2, Query2_response, reranked_chunks2)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


{'groundedness': {'score': 5,
  'reason': 'Answer is directly supported by the context.',
  'unsupported_claims': 'none'},
 'relevance': {'score': 5,
  'reason': 'Directly answers all aspects of the question.',
  'missing_aspects': 'none'},
 'faithfulness': {'is_faithful': False,
  'hallucinated_claims': 'IV fluids and antibiotics as treatment for non-perforated appendicitis, antibiotics not curative in cases where surgery is impossible',
  'severity': 'minor'}}

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
evaluate_with_judge(judge_llm,Query3, Query3_response, reranked_chunks3)

Loaded existing collection with 8200 chunks


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


{'groundedness': {'score': 3,
  'reason': 'The context does not directly discuss alopecia areata or its causes and treatments. However, the answer mentions some relevant information about this condition.',
  'unsupported_claims': 'none'},
 'relevance': {'score': 5,
  'reason': 'Directly addresses the question by providing information on treatments for sudden patchy hair loss and possible causes.',
  'missing_aspects': 'none'},
 'faithfulness': {'is_faithful': True,
  'hallucinated_claims': 'none',
  'severity': 'none'}}

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
evaluate_with_judge(judge_llm,Query4, Query4_response, reranked_chunks4)

Loaded existing collection with 8200 chunks


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


{'groundedness': {'score': 5,
  'reason': 'The answer directly repeats the information from the context about the recommended treatments for traumatic brain injury.',
  'unsupported_claims': 'none'},
 'relevance': {'score': 5,
  'reason': 'Directly and completely addresses the question by detailing the recommended treatments for a person with a brain injury.',
  'missing_aspects': 'none'},
 'faithfulness': {'is_faithful': True,
  'hallucinated_claims': 'none',
  'severity': 'none'}}

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
evaluate_with_judge(judge_llm,Query5, Query5_response, reranked_chunks5)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


{'groundedness': {'score': 2,
  'reason': 'Answer is loosely related to context but makes unsupported claims',
  'unsupported_claims': 'The answer does not specifically address a fractured leg during a hiking trip, and some treatment protocols mentioned are not directly applicable to this scenario.'},
 'relevance': {'score': 2,
  'reason': 'The answer partially addresses the question by providing general information about treating various types of injuries and fractures, but does not specifically address a fractured leg during a hiking trip.',
  'missing_aspects': 'Specific treatment protocol for a fractured leg during a hiking trip'},
 'faithfulness': {'is_faithful': False,
  'hallucinated_claims': 'none',
  'severity': 'minor'}}

| Query | Question | Groundedness | Relevance | Faithfulness | Overall |
|---|---|---|---|---|---|
| Q1 | Managing sepsis in critical care unit | 5/5 | 5/5 | False (minor) | Minor faithfulness concern |
| Q2 | Symptoms & treatment for appendicitis | 5/5 | 5/5 | False (minor) | Minor hallucination detected |
| Q3 | Sudden patchy hair loss treatments | 3/5 | 5/5 | True | Retrieval mismatch |
| Q4 | Treatments for traumatic brain injury | 5/5 | 5/5 | True | Best performing |
| Q5 | Precautions for fractured leg hiking | 2/5 | 2/5 | False (minor) | Retrieval failure |

## Actionable Insights and Business Recommendations

1. Retrieval-Augmented Generation (RAG) significantly improves medical answer accuracy

Experiments show that a standalone LLM without context produces generic responses, occasional hallucinations, missing medical references

After implementing RAG with medical documents (Merck Manuals) responses become more precise answers include domain-specific terminology and hallucination risk is reduced.

**Insight**: Healthcare AI systems must be grounded in trusted medical sources rather than relying solely on pretrained model knowledge.

**Impact**: Higher reliability of clinical information, Reduced risk of misinformation, and better trust from medical professionals

2. Document chunking and embeddings directly affect information retrieval quality

Experiments with chunking and embedding configurations demonstrates that semantic chunking + embedding models allow the system to retrieve relevant medical passages before generating an answer.

Poor chunking can lead to incomplete context, irrelevant retrieval, and weaker answers

**Insight**: The quality of the retrieval layer is as important as the LLM itself.

**Impact**: To build a medical AI assistants we must invest in effort high-quality document preprocessing optimized chunking strategies and domain-specific embedding models

3. Re-Ranking improves context relevance for complex medical queries

By introducing cross-encoder re-ranking, the system filters retrieved chunks to ensure the most relevant medical passages are used. Without re-ranking vector search may retrieve partially relevant documents

With re-ranking, higher contextual precision and more accurate answers can be achieved

**Insight**: Multi-stage retrieval pipelines significantly improve AI reliability for complex knowledge domains like healthcare.

**Impact**: This architecture reduces the chance that incorrect or irrelevant passages influence the final answer.

4. LLM Prompt Engineering Improves Safety and Response Quality

**prompt engineering** helps ensure that the system uses only retrieved context,that hallucinations are minimized and answers follow structured medical explanations

**Insight**: Prompt engineering is a critical control layer for medical AI safety.

**Impact**: Organizations deploying AI assistants should implement strict prompting frameworks that enforce, context grounding, structured answers and source citations

## **Business Recommendations**

1. Deploy AI-Powered Medical knowledge assistants in clinical settings

Healthcare organizations should deploy RAG-based medical assistants that provide quick access to clinical knowledge, summarized treatment guidelines, evidence-based medical information

**Business Value**: Improved physician productivity, Reduced research time, Better patient care outcomes

2. Integrate Medical AI Assistants Into Clinical Workflow Systems

The assistant should be integrated with systems such as Electronic Health Records (EHR), Clinical decision support systems Hospital knowledge portals

**Business Value**: Doctors can retrieve medical information without leaving their workflow environment.

3. Use curated medical knowledge bases instead of open internet sources

Healthcare AI systems should rely on trusted sources such as Merck Manuals

PubMed, Clinical guidelines, peer-reviewed literature

**Business Value**: Improved reliability regulatory compliance and reduced liability risk

4. Implement Multi-Layer safety controls for Medical AI

Before deploying AI in healthcare, organizations should implement grounded RAG architecture, safety prompts, hallucination detection, and  human oversight

**Business Value**: improved regulatory compliance, safer clinical recommendations increased trust from healthcare providers

5. Continuously update the medical knowledge base

Medical guidelines change frequently. The system should support continuous ingestion of new medical literature, automated indexing of updated guidelines

**Business Value**: Healthcare professionals always access to the current medical knowledge.

<font size=6 color='blue'>Power Ahead</font>
___